Cell 1. 기본 경로/토큰/환경변수 설정

Cell 2. 기본 경로/토큰 / 환경변수 print 확인

Cell 3. AnimatedDrawings 여부 > /kaggle/working/AnimatedDrawings 없을시 clone 진행 

Cell 4. USE_MESA patch

Cell 5. kaggle/working/AnidatedDrawings 폴더 구조 print 확인

Cell 6. dagshub mlflow 기록 + 6개 낙서 렌더링 실행 (image_to_animation.py 실행, annotations_to_animation.py 실행,stdout/stderr 출력,video.gif 존재 여부 확인)

Cell 7. 낱개 6개 GIF + 합본 GIF 표시

Cell 8. TorchServe 시작 : DagsHub + OSMesa + 6개 액션 worker.py 생성 후 Celery 시작

In [ ]:
! pip install mlflow

In [ ]:
from pathlib import Path
import os, subprocess
from kaggle_secrets import UserSecretsClient


DAGSHUB_USERNAME = "myelin24m"
DAGSHUB_REPO = "Kride"  # .mlflow 붙이지 않기
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

os.environ["DAGSHUB_USER"] = DAGSHUB_USERNAME
os.environ["DAGSHUB_REPO"] = DAGSHUB_REPO
os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = (
    f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
)

WORK_DIR = "/kaggle/working/AnimatedDrawings"
PYTHON_EXE = "/usr/bin/python3"  # 또는 "/kaggle/working/miniconda/envs/ad_env/bin/python"

image_path = "/kaggle/working/AnimatedDrawings/examples/drawings/garlic.png"
out_dir = "/kaggle/working/test_garlic_out"

motion_path = "/kaggle/working/AnimatedDrawings/examples/config/motion/dab.yaml"
retarget_path = "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml"

Path(out_dir).mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = WORK_DIR

BASE_DIR = "/kaggle/working/AnimatedDrawings"
print("AnimatedDrawings:", Path(BASE_DIR).exists())
print("motion yaml:", len(list(Path(BASE_DIR, "examples/config/motion").glob("*.yaml"))))
print("retarget yaml:", len(list(Path(BASE_DIR, "examples/config/retarget").glob("*.yaml"))))
print("DagsHub token:", bool(os.environ.get("DAGSHUB_TOKEN")))
print("MLflow URI:", os.environ.get("MLFLOW_TRACKING_URI"))
print('쉘1 끝')

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/kaggle/working/AnimatedDrawings")

if BASE_DIR.exists():
    print("✅ AnimatedDrawings already exists:", BASE_DIR)
else:
    print("▶️ AnimatedDrawings clone 중...")
    os.system("git clone https://github.com/facebookresearch/AnimatedDrawings.git /kaggle/working/AnimatedDrawings")

print("존재 여부:", BASE_DIR.exists())
print("examples 존재 여부:", (BASE_DIR / "examples").exists())

BASE_DIR = Path("/kaggle/working/AnimatedDrawings")
anno = BASE_DIR / "examples" / "annotations_to_animation.py"

src = anno.read_text()

if "'view': {'USE_MESA': True" not in src and '"view": {"USE_MESA": True' not in src:
    src = src.replace(
        "'controller': {",
        "'view': {'USE_MESA': True},\n        'controller': {",
        1,
    )
    anno.write_text(src)
    print("✅ USE_MESA patch 완료")
else:
    print("✅ USE_MESA patch 이미 적용됨")
    
print('쉘2 끝')

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path("/kaggle/working/AnimatedDrawings")

print("BASE:", BASE_DIR.exists())
print("characters:", list((BASE_DIR / "examples" / "characters").glob("char*"))[:6])
print("motions:", list((BASE_DIR / "examples" / "config" / "motion").glob("*.yaml")))
print("retargets:", list((BASE_DIR / "examples" / "config" / "retarget").glob("*.yaml")))
print('쉘3 끝')

In [ ]:
import os, subprocess, time
from pathlib import Path
# miniconda → ad_env → torchserve 설치 순서가 반드시 필요
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/ad_env"
PYTHON_EXE = f"{ENV_DIR}/bin/python"
PIP_EXE = f"{ENV_DIR}/bin/pip"
TORCHSERVE = f"{ENV_DIR}/bin/torchserve"

# 1. Miniconda 설치
if not Path(CONDA_DIR).exists():
    print("▶️ Miniconda 설치 중...")
    os.chdir("/kaggle/working")
    os.system("wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh")
    os.system("bash Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -b -p /kaggle/working/miniconda")
else:
    print("✅ Miniconda already exists")

# 2. Python 3.8 환경 생성
if not Path(ENV_DIR).exists():
    print("▶️ ad_env 생성 중...")
    os.system(f"{CONDA_DIR}/bin/conda create -n ad_env python=3.8 -y")
else:
    print("✅ ad_env already exists")

print("python exists:", Path(PYTHON_EXE).exists())
import os
from pathlib import Path

ENV_DIR = "/kaggle/working/miniconda/envs/ad_env"
PYTHON_EXE = f"{ENV_DIR}/bin/python"
PIP_EXE = f"{ENV_DIR}/bin/pip"
TORCHSERVE = f"{ENV_DIR}/bin/torchserve"

# 시스템 OSMesa
os.system("apt-get update -qq")
os.system("apt-get install -y -qq libosmesa6 libosmesa6-dev freeglut3-dev libglfw3-dev")

# pip 기본 정리
os.system(f"{PIP_EXE} install --upgrade pip setuptools==65.5.0 wheel six --quiet")

# AnimatedDrawings / TorchServe 쪽 핵심
os.system(f"{PIP_EXE} install chumpy --no-build-isolation --quiet")
os.system(f"{PIP_EXE} install torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu116 --quiet")
os.system(f"{PIP_EXE} install mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu116/torch1.13.0/index.html --quiet")
os.system(f"{PIP_EXE} install mmdet==2.28.2 mmpose==0.28.1 --quiet")
os.system(f"{PIP_EXE} install torchserve==0.8.0 torch-model-archiver==0.8.0 --quiet")
os.system(f"{PIP_EXE} install scikit-image scikit-learn PyOpenGL==3.1.7 PyOpenGL-accelerate imageio imageio-ffmpeg shapely pillow pyyaml requests mlflow dagshub --quiet")

print("python exists:", Path(PYTHON_EXE).exists())
print("torchserve exists:", Path(TORCHSERVE).exists())

print('쉘5 끝')

In [ ]:
import os, subprocess, time
from pathlib import Path

ENV_DIR = "/kaggle/working/miniconda/envs/ad_env"
PYTHON_EXE = f"{ENV_DIR}/bin/python"
PIP_EXE = f"{ENV_DIR}/bin/pip"
TORCHSERVE = f"{ENV_DIR}/bin/torchserve"
import subprocess

os.system(f"{PIP_EXE} install torchserve==0.8.0 torch-model-archiver==0.8.0 --quiet")
os.system(f"{PIP_EXE} install PyOpenGL==3.1.7 imageio imageio-ffmpeg pillow pyyaml requests --quiet")

print("python exists:", Path(PYTHON_EXE).exists())
print("torchserve exists:", Path(TORCHSERVE).exists())
print(subprocess.check_output([PYTHON_EXE, "--version"], text=True))
print(subprocess.check_output([TORCHSERVE, "--version"], text=True))
print('쉘6 끝')

 6개 낙서 + 6개 액션 worker 셀을 실행하면 됩니다. 다만 새 노트북에서는 렌더링 안정성을 위해 worker 안의 이 부분은 꼭 이렇게 가세요.

In [ ]:
os.system("apt-get update -qq")
os.system("apt-get install -y -qq libosmesa6 libosmesa6-dev freeglut3-dev libglfw3-dev")

In [ ]:
import shutil, os
from pathlib import Path

BASE_DIR = "/kaggle/working/AnimatedDrawings"
MODEL_STORE = Path(BASE_DIR) / "torchserve" / "model_store"
MODEL_STORE.mkdir(parents=True, exist_ok=True)

def find_and_copy_model(keyword, dest_name):
    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if keyword in file and file.endswith(".mar"):
                src = Path(root) / file
                dst = MODEL_STORE / dest_name
                shutil.copy(src, dst)
                print("copied:", src, "->", dst)
                return True
    print("not found:", keyword)
    return False

find_and_copy_model("detector", "drawn_humanoid_detector.mar")
find_and_copy_model("pose", "drawn_humanoid_pose_estimator.mar")

print(list(MODEL_STORE.glob("*.mar")))

In [ ]:
import os, subprocess, time
from pathlib import Path

BASE_DIR = "/kaggle/working/AnimatedDrawings"
TORCHSERVE = "/kaggle/working/miniconda/envs/ad_env/bin/torchserve"

config_path = Path(BASE_DIR) / "torchserve" / "config.properties"
config_path.write_text(
    "inference_address=http://0.0.0.0:8080\n"
    "management_address=http://0.0.0.0:8081\n"
    "metrics_address=http://0.0.0.0:8082\n"
    "enable_metrics_api=false\n"
    "default_response_timeout=600\n"
)

os.system(f"{TORCHSERVE} --stop >/dev/null 2>&1")
os.system("pkill -9 -f torchserve")
os.system("pkill -9 -f java")
time.sleep(2)

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["TS_STARTUP_TIMEOUT"] = "600"

cmd = [
    TORCHSERVE,
    "--start",
    "--ncs",
    "--model-store", "torchserve/model_store",
    "--ts-config", "torchserve/config.properties",
    "--models",
    "drawn_humanoid_detector=drawn_humanoid_detector.mar",
    "drawn_humanoid_pose_estimator=drawn_humanoid_pose_estimator.mar",
]

subprocess.Popen(
    cmd,
    cwd=BASE_DIR,
    env=env,
    stdout=open("/kaggle/working/torchserve.log", "a"),
    stderr=subprocess.STDOUT,
)

print("⏳ TorchServe loading...")
time.sleep(60)
print(Path("/kaggle/working/torchserve.log").read_text()[-2000:])
print('쉘8 끝')

In [ ]:
import subprocess

PIP_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/pip"

subprocess.run([
    PIP_EXE, "install", "-q",
    "tqdm",
    "matplotlib",
    "opencv-python-headless",
    "scikit-image",
    "scikit-learn",
    "Pillow",
    "PyYAML",
    "shapely",
    "imageio",
    "imageio-ffmpeg"
], check=False)

print("✅ ad_env 렌더링 보조 패키지 설치 완료")

In [ ]:
import os, subprocess
from pathlib import Path

WORK_DIR = "/kaggle/working/AnimatedDrawings"
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"

image_path = "/kaggle/working/AnimatedDrawings/examples/drawings/garlic.png"
out_dir = "/kaggle/working/test_garlic_out"
motion_path = "/kaggle/working/AnimatedDrawings/examples/config/motion/dab.yaml"
retarget_path = "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml"

Path(out_dir).mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = WORK_DIR

cmd_annotate = [
    PYTHON_EXE,
    f"{WORK_DIR}/examples/image_to_animation.py",
    image_path,
    out_dir,
]

annotate_result = subprocess.run(
    cmd_annotate,
    env=env,
    cwd=WORK_DIR,
    capture_output=True,
    text=True,
)

print("annotate return:", annotate_result.returncode)
print(annotate_result.stderr[-1000:])

cmd_render = [
    PYTHON_EXE,
    f"{WORK_DIR}/examples/annotations_to_animation.py",
    out_dir,
    motion_path,
    retarget_path,
]

result = subprocess.run(
    cmd_render,
    env=env,
    cwd=WORK_DIR,
    capture_output=True,
    text=True,
)

print("render return:", result.returncode)
print(result.stderr[-1000:])
print("GIF:", Path(out_dir, "video.gif").exists())
print("쉘 9끝")

In [ ]:
import os, time, subprocess
from pathlib import Path
import mlflow
from IPython.display import Image as IPyImage, display

WORK_DIR = "/kaggle/working/AnimatedDrawings"
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"

char_dir = "/kaggle/working/AnimatedDrawings/examples/characters/char1"
motion_path = "/kaggle/working/AnimatedDrawings/examples/config/motion/dab.yaml"
retarget_path = "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("animated-drawings-pre-rigged-test")

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = WORK_DIR

started = time.time()

with mlflow.start_run(run_name="char1-dab-render") as run:
    mlflow.log_param("char_dir", char_dir)
    mlflow.log_param("motion_path", motion_path)
    mlflow.log_param("retarget_path", retarget_path)

    cmd = [
        PYTHON_EXE,
        f"{WORK_DIR}/examples/annotations_to_animation.py",
        char_dir,
        motion_path,
        retarget_path,
    ]

    result = subprocess.run(
        cmd,
        env=env,
        cwd=WORK_DIR,
        capture_output=True,
        text=True,
    )

    Path(char_dir, "render_stdout.txt").write_text(result.stdout[-10000:])
    Path(char_dir, "render_stderr.txt").write_text(result.stderr[-10000:])

    gif_path = Path(char_dir, "video.gif")
    success = gif_path.exists()

    mlflow.log_metric("render_return_code", result.returncode)
    mlflow.log_metric("success", 1 if success else 0)
    mlflow.log_metric("elapsed_seconds", time.time() - started)
    mlflow.log_artifacts(char_dir, artifact_path="char1_output")

    print("render return:", result.returncode)
    print(result.stderr[-1000:])
    print("GIF:", success)
    print("DagsHub run id:", run.info.run_id)
    print("DagsHub uri:", mlflow.get_tracking_uri())

if gif_path.exists():
    display(IPyImage(filename=str(gif_path)))

print("쉘10 끝")

이렇게 하는 이유는 image_to_animation.py는 기본적으로 이미지 경로 + 출력 폴더만 안정적으로 받는 구조이고, 다른 액션은 annotations_to_animation.py에서 motion_cfg, retarget_cfg를 바꿔 적용하는 게 맞습니다.

그리고 표시 셀은 그대로 실행하면 됩니다.

In [ ]:
# animation 최종본 1

import os, time, subprocess, base64, html
from pathlib import Path
import mlflow
from PIL import Image, ImageSequence, ImageDraw
from IPython.display import HTML, Image as IPyImage, display

WORK_DIR = "/kaggle/working/AnimatedDrawings"
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"

char_dirs = [
    "/kaggle/working/AnimatedDrawings/examples/characters/char1",
    "/kaggle/working/AnimatedDrawings/examples/characters/char2",
    "/kaggle/working/AnimatedDrawings/examples/characters/char3",
    "/kaggle/working/AnimatedDrawings/examples/characters/char4",
    "/kaggle/working/AnimatedDrawings/examples/characters/char5",
    "/kaggle/working/AnimatedDrawings/examples/characters/char6",
]

motions = [
    "/kaggle/working/AnimatedDrawings/examples/config/motion/dab.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/motion/wave_hello.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/motion/jumping.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/motion/jumping_jacks.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/motion/zombie.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/motion/jesse_dance.yaml",
]

retargets = [
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/cmu1_pfp.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml",
    "/kaggle/working/AnimatedDrawings/examples/config/retarget/fair1_ppf.yaml",
]

ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions")
ROOT_OUTPUT.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = WORK_DIR

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("animated-drawings-six-actions")

def copy_char(src_dir, dst_dir):
    src = Path(src_dir)
    dst = Path(dst_dir)
    if dst.exists():
        import shutil
        shutil.rmtree(dst)
    import shutil
    shutil.copytree(src, dst)
    return dst

def make_combined_gif(items, output_path, cols=3):
    valid = [item for item in items if item["gif_path"] and Path(item["gif_path"]).exists()]
    if not valid:
        return None

    cell_w, cell_h, label_h = 280, 280, 34
    rows = (len(valid) + cols - 1) // cols
    canvas_size = (cols * cell_w, rows * (cell_h + label_h))

    all_frames = []
    max_frames = 0
    duration = 80

    for item in valid:
        im = Image.open(item["gif_path"])
        duration = im.info.get("duration", duration)
        frames = []
        for frame in ImageSequence.Iterator(im):
            fr = frame.convert("RGBA")
            fr.thumbnail((cell_w, cell_h), Image.LANCZOS)
            bg = Image.new("RGBA", (cell_w, cell_h), "white")
            bg.alpha_composite(fr, ((cell_w - fr.width) // 2, (cell_h - fr.height) // 2))
            frames.append(bg.convert("RGB"))
        frames = frames[:180]
        max_frames = max(max_frames, len(frames))
        all_frames.append((item, frames))

    output_frames = []
    for frame_idx in range(max_frames):
        canvas = Image.new("RGB", canvas_size, "white")
        draw = ImageDraw.Draw(canvas)

        for i, (item, frames) in enumerate(all_frames):
            x = (i % cols) * cell_w
            y = (i // cols) * (cell_h + label_h)
            canvas.paste(frames[frame_idx % len(frames)], (x, y))
            draw.text((x + 8, y + cell_h + 8), f"{item['index']}. {item['action']}", fill="black")

        output_frames.append(canvas)

    output_frames[0].save(
        output_path,
        save_all=True,
        append_images=output_frames[1:],
        duration=duration,
        loop=0,
    )
    return str(output_path)

started = time.time()
results = []

with mlflow.start_run(run_name="six-pre-rigged-actions") as run:
    for i, (char_dir, motion, retarget) in enumerate(zip(char_dirs, motions, retargets), 1):
        action = Path(motion).stem
        out_dir = ROOT_OUTPUT / f"{i:02d}_{action}"
        copy_char(char_dir, out_dir)

        print(f"▶️ {i}/6 렌더링: {Path(char_dir).name} + {action}")

        cmd = [
            PYTHON_EXE,
            f"{WORK_DIR}/examples/annotations_to_animation.py",
            str(out_dir),
            motion,
            retarget,
        ]

        result = subprocess.run(
            cmd,
            env=env,
            cwd=WORK_DIR,
            capture_output=True,
            text=True,
        )

        Path(out_dir, "render_stdout.txt").write_text(result.stdout[-10000:])
        Path(out_dir, "render_stderr.txt").write_text(result.stderr[-10000:])

        gif_path = out_dir / "video.gif"
        ok = gif_path.exists()

        results.append({
            "index": i,
            "action": action,
            "gif_path": str(gif_path) if ok else None,
            "error": result.stderr[-1000:] if not ok else "",
            "return_code": result.returncode,
        })

        mlflow.log_param(f"action_{i}", action)
        mlflow.log_metric(f"return_code_{i}", result.returncode)
        mlflow.log_metric(f"success_{i}", 1 if ok else 0)

        print("   return:", result.returncode, "gif:", ok)

    combined_path = ROOT_OUTPUT / "combined_2x3.gif"
    combined = make_combined_gif(results, combined_path)

    mlflow.log_metric("success_count", sum(1 for r in results if r["gif_path"]))
    mlflow.log_metric("elapsed_seconds", time.time() - started)
    mlflow.log_artifacts(str(ROOT_OUTPUT), artifact_path="six_doodle_actions")

    print("DagsHub run id:", run.info.run_id)
    print("DagsHub uri:", mlflow.get_tracking_uri())

def gif_to_data_uri(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/gif;base64,{data}"

cells = []
for item in results:
    if item["gif_path"] and os.path.exists(item["gif_path"]):
        cells.append(f"""
        <td style="padding:10px;text-align:center;vertical-align:top">
          <div style="font-weight:600;margin-bottom:6px">{item["index"]}. {html.escape(item["action"])}</div>
          <img src="{gif_to_data_uri(item["gif_path"])}" width="220">
        </td>
        """)
    else:
        cells.append(f"""
        <td style="padding:10px;color:#b00020;vertical-align:top">
          <b>{item["index"]}. {html.escape(item["action"])}</b>
          <pre>{html.escape(item["error"][:500])}</pre>
        </td>
        """)

rows = ["<tr>" + "".join(cells[i:i+3]) + "</tr>" for i in range(0, len(cells), 3)]
display(HTML("<table>" + "".join(rows) + "</table>"))

if combined and os.path.exists(combined):
    print("🎬 합본 GIF")
    display(IPyImage(filename=combined))
else:
    print("❌ 합본 GIF 생성 실패")

print("쉘11 끝")

In [ ]:
from pathlib import Path
print("실패원인")
for p in [
    "/kaggle/working/six_doodle_actions/05_zombie/render_stderr.txt",
    "/kaggle/working/six_doodle_actions/06_jesse_dance/render_stderr.txt",
]:
    print("=" * 80)
    print(p)
    if Path(p).exists():
        print(Path(p).read_text()[-2000:])
    else:
        print("없음")

In [ ]:

import os, time, subprocess, shutil, json, base64, html, re
from pathlib import Path
import mlflow
from PIL import Image, ImageSequence, ImageDraw
from IPython.display import HTML, Image as IPyImage, display

WORK_DIR = Path("/kaggle/working/AnimatedDrawings")
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"
ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions_fallback")
ROOT_OUTPUT.mkdir(parents=True, exist_ok=True)

# 필요 패키지 보강
subprocess.run([
    "/kaggle/working/miniconda/envs/ad_env/bin/pip",
    "install", "-q", "tqdm", "pillow", "pyyaml", "imageio", "imageio-ffmpeg"
], check=False)

# OSMesa patch
anno = WORK_DIR / "examples" / "annotations_to_animation.py"
src = anno.read_text()
if "'view': {'USE_MESA': True" not in src and '"view": {"USE_MESA": True' not in src:
    src = src.replace(
        "'controller': {",
        "'view': {'USE_MESA': True},\n        'controller': {",
        1,
    )
    anno.write_text(src)
    print("✅ USE_MESA patch 완료")

def p(*parts):
    return str(WORK_DIR.joinpath(*parts))

CHAR = {
    "char1": p("examples", "characters", "char1"),
    "char2": p("examples", "characters", "char2"),
    "char3": p("examples", "characters", "char3"),
    "char4": p("examples", "characters", "char4"),
    "char5": p("examples", "characters", "char5"),
    "char6": p("examples", "characters", "char6"),
}

MOTION = {
    "dab": p("examples", "config", "motion", "dab.yaml"),
    "wave_hello": p("examples", "config", "motion", "wave_hello.yaml"),
    "jumping": p("examples", "config", "motion", "jumping.yaml"),
    "jumping_jacks": p("examples", "config", "motion", "jumping_jacks.yaml"),
    "zombie": p("examples", "config", "motion", "zombie.yaml"),
    "jesse_dance": p("examples", "config", "motion", "jesse_dance.yaml"),
}

RETARGET = {
    "fair1_ppf": p("examples", "config", "retarget", "fair1_ppf.yaml"),
    "cmu1_pfp": p("examples", "config", "retarget", "cmu1_pfp.yaml"),
    "mixamo_fff": p("examples", "config", "retarget", "mixamo_fff.yaml"),
}

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = str(WORK_DIR)

def char_has_joint(char_dir, joint_name):
    cfg = Path(char_dir) / "char_cfg.yaml"
    if not cfg.exists():
        return False
    text = cfg.read_text(errors="ignore").lower()
    return joint_name.lower() in text

def parse_missing_joint(stderr):
    m = re.search(r"Could not find joint1 in runtime check:\s*([A-Za-z0-9_]+)", stderr)
    return m.group(1) if m else None

def copy_character(src_char_dir, dst_dir):
    dst = Path(dst_dir)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_char_dir, dst)
    return dst

def render_once(src_char_dir, motion_path, retarget_path, attempt_dir):
    attempt_dir = copy_character(src_char_dir, attempt_dir)

    # precheck: humanoid retarget 대부분은 shoulder가 필요함
    if not char_has_joint(attempt_dir, "right_shoulder"):
        msg = "precheck_missing_joint:right_shoulder"
        (attempt_dir / "render_stderr.txt").write_text(msg)
        return {
            "success": False,
            "return_code": None,
            "gif_path": None,
            "stderr": msg,
            "missing_joint": "right_shoulder",
            "attempt_dir": str(attempt_dir),
        }

    cmd = [
        PYTHON_EXE,
        str(WORK_DIR / "examples" / "annotations_to_animation.py"),
        str(attempt_dir),
        str(motion_path),
        str(retarget_path),
    ]

    result = subprocess.run(
        cmd,
        env=env,
        cwd=str(WORK_DIR),
        capture_output=True,
        text=True,
    )

    (attempt_dir / "render_stdout.txt").write_text(result.stdout[-10000:])
    (attempt_dir / "render_stderr.txt").write_text(result.stderr[-10000:])

    gif_path = attempt_dir / "video.gif"
    success = gif_path.exists()

    return {
        "success": success,
        "return_code": result.returncode,
        "gif_path": str(gif_path) if success else None,
        "stderr": result.stderr[-2000:],
        "missing_joint": parse_missing_joint(result.stderr),
        "attempt_dir": str(attempt_dir),
    }

def build_attempts(character, requested_motion, requested_retarget):
    requested_name = Path(requested_motion).stem

    attempts = [
        {
            "character": character,
            "motion": requested_motion,
            "retarget": requested_retarget,
            "reason": "requested",
        }
    ]

    motion_specific_retargets = {
        "jumping_jacks": [RETARGET["cmu1_pfp"], RETARGET["fair1_ppf"]],
        "jesse_dance": [RETARGET["mixamo_fff"], RETARGET["fair1_ppf"]],
        "zombie": [RETARGET["fair1_ppf"], RETARGET["mixamo_fff"]],
    }

    for retarget in motion_specific_retargets.get(requested_name, []):
        attempts.append({
            "character": character,
            "motion": requested_motion,
            "retarget": retarget,
            "reason": f"same_character_alt_retarget:{Path(retarget).stem}",
        })

    # 캐릭터 skeleton이 안 맞으면 안정 캐릭터로 같은 motion 재시도
    for safe_char in [CHAR["char1"], CHAR["char2"], CHAR["char3"], CHAR["char4"]]:
        attempts.append({
            "character": safe_char,
            "motion": requested_motion,
            "retarget": requested_retarget,
            "reason": f"safe_character_same_motion:{Path(safe_char).name}",
        })

    # 끝까지 안 되면 사용자에게 빈 칸 대신 보여줄 안전 motion
    for safe_char, safe_motion in [
        (CHAR["char1"], MOTION["wave_hello"]),
        (CHAR["char2"], MOTION["dab"]),
    ]:
        attempts.append({
            "character": safe_char,
            "motion": safe_motion,
            "retarget": RETARGET["fair1_ppf"],
            "reason": f"last_resort:{Path(safe_char).name}+{Path(safe_motion).stem}",
        })

    # 중복 제거
    deduped = []
    seen = set()
    for a in attempts:
        key = (a["character"], a["motion"], a["retarget"])
        if key not in seen:
            seen.add(key)
            deduped.append(a)
    return deduped

def render_with_fallback(index, character, requested_motion, requested_retarget):
    action = Path(requested_motion).stem
    attempts = build_attempts(character, requested_motion, requested_retarget)

    all_attempts = []
    for attempt_no, attempt in enumerate(attempts, 1):
        attempt_dir = ROOT_OUTPUT / f"{index:02d}_{action}" / f"attempt_{attempt_no:02d}_{attempt['reason'].replace(':', '_')}"
        print(f"   ↳ attempt {attempt_no}: {Path(attempt['character']).name} + {Path(attempt['motion']).stem} + {Path(attempt['retarget']).stem}")

        r = render_once(
            attempt["character"],
            attempt["motion"],
            attempt["retarget"],
            attempt_dir,
        )

        record = {
            **attempt,
            **r,
            "attempt_no": attempt_no,
        }
        all_attempts.append(record)

        if r["success"]:
            return {
                "index": index,
                "requested_action": action,
                "action": Path(attempt["motion"]).stem,
                "character": Path(attempt["character"]).name,
                "motion": attempt["motion"],
                "retarget": attempt["retarget"],
                "gif_path": r["gif_path"],
                "success": True,
                "fallback_used": attempt_no > 1,
                "fallback_reason": attempt["reason"],
                "attempts": all_attempts,
                "error": "",
                "missing_joint": None,
            }

    last = all_attempts[-1]
    return {
        "index": index,
        "requested_action": action,
        "action": action,
        "character": Path(character).name,
        "motion": requested_motion,
        "retarget": requested_retarget,
        "gif_path": None,
        "success": False,
        "fallback_used": True,
        "fallback_reason": "all_attempts_failed",
        "attempts": all_attempts,
        "error": last.get("stderr", "")[:1000],
        "missing_joint": last.get("missing_joint"),
    }

def make_combined_gif(items, output_path, cols=3):
    valid = [item for item in items if item.get("gif_path") and Path(item["gif_path"]).exists()]
    if not valid:
        return None

    cell_w, cell_h, label_h = 280, 280, 34
    rows = (len(valid) + cols - 1) // cols
    canvas_size = (cols * cell_w, rows * (cell_h + label_h))

    all_frames = []
    max_frames = 0
    duration = 80

    for item in valid:
        im = Image.open(item["gif_path"])
        duration = im.info.get("duration", duration)
        frames = []
        for frame in ImageSequence.Iterator(im):
            fr = frame.convert("RGBA")
            fr.thumbnail((cell_w, cell_h), Image.LANCZOS)
            bg = Image.new("RGBA", (cell_w, cell_h), "white")
            bg.alpha_composite(fr, ((cell_w - fr.width) // 2, (cell_h - fr.height) // 2))
            frames.append(bg.convert("RGB"))
        frames = frames[:180]
        max_frames = max(max_frames, len(frames))
        all_frames.append((item, frames))

    output_frames = []
    for frame_idx in range(max_frames):
        canvas = Image.new("RGB", canvas_size, "white")
        draw = ImageDraw.Draw(canvas)

        for i, (item, frames) in enumerate(all_frames):
            x = (i % cols) * cell_w
            y = (i // cols) * (cell_h + label_h)
            canvas.paste(frames[frame_idx % len(frames)], (x, y))

            label = f"{item['index']}. {item['requested_action']}"
            if item.get("fallback_used"):
                label += f" -> {item['action']}"
            draw.text((x + 8, y + cell_h + 8), label, fill="black")

        output_frames.append(canvas)

    output_frames[0].save(
        output_path,
        save_all=True,
        append_images=output_frames[1:],
        duration=duration,
        loop=0,
    )
    return str(output_path)

jobs = [
    (CHAR["char1"], MOTION["dab"], RETARGET["fair1_ppf"]),
    (CHAR["char2"], MOTION["wave_hello"], RETARGET["fair1_ppf"]),
    (CHAR["char3"], MOTION["jumping"], RETARGET["fair1_ppf"]),
    (CHAR["char4"], MOTION["jumping_jacks"], RETARGET["cmu1_pfp"]),
    (CHAR["char5"], MOTION["zombie"], RETARGET["fair1_ppf"]),
    (CHAR["char6"], MOTION["jesse_dance"], RETARGET["mixamo_fff"]),
]

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:///kaggle/working/mlruns"))
mlflow.set_experiment("animated-drawings-six-actions-fallback")

started = time.time()
results = []

with mlflow.start_run(run_name="six-actions-render-with-fallback") as run:
    for index, (character, motion, retarget) in enumerate(jobs, 1):
        print(f"▶️ {index}/6 요청: {Path(character).name} + {Path(motion).stem}")
        item = render_with_fallback(index, character, motion, retarget)
        results.append(item)

        mlflow.log_param(f"requested_action_{index}", item["requested_action"])
        mlflow.log_param(f"actual_action_{index}", item["action"])
        mlflow.log_param(f"fallback_used_{index}", item["fallback_used"])
        mlflow.log_param(f"fallback_reason_{index}", item["fallback_reason"])
        mlflow.log_metric(f"success_{index}", 1 if item["success"] else 0)
        mlflow.log_metric(f"attempt_count_{index}", len(item["attempts"]))

        print(f"   result: success={item['success']} fallback={item['fallback_used']} gif={bool(item['gif_path'])}")

    combined_path = ROOT_OUTPUT / "combined_2x3.gif"
    combined = make_combined_gif(results, combined_path)

    summary_path = ROOT_OUTPUT / "render_summary.json"
    summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

    mlflow.log_metric("success_count", sum(1 for r in results if r["success"]))
    mlflow.log_metric("elapsed_seconds", time.time() - started)
    mlflow.log_artifacts(str(ROOT_OUTPUT), artifact_path="six_actions_with_fallback")

    print("DagsHub run id:", run.info.run_id)
    print("DagsHub uri:", mlflow.get_tracking_uri())

def gif_to_data_uri(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/gif;base64,{data}"

cells = []
for item in results:
    if item["gif_path"] and os.path.exists(item["gif_path"]):
        tag = "fallback" if item["fallback_used"] else "original"
        cells.append(f"""
        <td style="padding:10px;text-align:center;vertical-align:top">
          <div style="font-weight:600;margin-bottom:6px">{item["index"]}. {html.escape(item["requested_action"])} ({tag})</div>
          <img src="{gif_to_data_uri(item["gif_path"])}" width="220">
        </td>
        """)
    else:
        cells.append(f"""
        <td style="padding:10px;color:#b00020;vertical-align:top">
          <b>{item["index"]}. {html.escape(item["requested_action"])}</b>
          <pre>{html.escape(item["error"][:500])}</pre>
        </td>
        """)

rows = ["<tr>" + "".join(cells[i:i+3]) + "</tr>" for i in range(0, len(cells), 3)]
display(HTML("<table>" + "".join(rows) + "</table>"))

if combined and os.path.exists(combined):
    print("🎬 합본 GIF")
    display(IPyImage(filename=combined))
else:
    print("❌ 합본 GIF 생성 실패")

In [ ]:
# animation 최종본 2


import os, time, subprocess, shutil, json, base64, html, re
from pathlib import Path

import mlflow
from PIL import Image, ImageSequence, ImageDraw
from IPython.display import HTML, Image as IPyImage, display

WORK_DIR = Path("/kaggle/working/AnimatedDrawings")
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"
PIP_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/pip"

ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions_fallback")
ROOT_OUTPUT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [PIP_EXE, "install", "-q", "tqdm", "pillow", "pyyaml", "imageio", "imageio-ffmpeg"],
    check=False,
)

anno = WORK_DIR / "examples" / "annotations_to_animation.py"
src = anno.read_text()
if "'view': {'USE_MESA': True" not in src and '"view": {"USE_MESA": True' not in src:
    src = src.replace(
        "'controller': {",
        "'view': {'USE_MESA': True},\n        'controller': {",
        1,
    )
    anno.write_text(src)
    print("✅ USE_MESA patch 완료")

def p(*parts):
    return str(WORK_DIR.joinpath(*parts))

CHAR = {
    "char1": p("examples", "characters", "char1"),
    "char2": p("examples", "characters", "char2"),
    "char3": p("examples", "characters", "char3"),
    "char4": p("examples", "characters", "char4"),
    "char5": p("examples", "characters", "char5"),
    "char6": p("examples", "characters", "char6"),
}

MOTION = {
    "dab": p("examples", "config", "motion", "dab.yaml"),
    "wave_hello": p("examples", "config", "motion", "wave_hello.yaml"),
    "jumping": p("examples", "config", "motion", "jumping.yaml"),
    "jumping_jacks": p("examples", "config", "motion", "jumping_jacks.yaml"),
    "zombie": p("examples", "config", "motion", "zombie.yaml"),
    "jesse_dance": p("examples", "config", "motion", "jesse_dance.yaml"),
}

RETARGET = {
    "fair1_ppf": p("examples", "config", "retarget", "fair1_ppf.yaml"),
    "cmu1_pfp": p("examples", "config", "retarget", "cmu1_pfp.yaml"),
    "mixamo_fff": p("examples", "config", "retarget", "mixamo_fff.yaml"),
}

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = str(WORK_DIR)

def char_has_joint(char_dir, joint_name):
    cfg = Path(char_dir) / "char_cfg.yaml"
    if not cfg.exists():
        return False
    return joint_name.lower() in cfg.read_text(errors="ignore").lower()

def parse_missing_joint(stderr):
    m = re.search(r"Could not find joint1 in runtime check:\s*([A-Za-z0-9_]+)", stderr)
    return m.group(1) if m else None

def copy_character(src_char_dir, dst_dir):
    dst = Path(dst_dir)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_char_dir, dst)
    return dst

def render_once(src_char_dir, motion_path, retarget_path, attempt_dir):
    attempt_dir = copy_character(src_char_dir, attempt_dir)

    if not char_has_joint(attempt_dir, "right_shoulder"):
        msg = "precheck_missing_joint:right_shoulder"
        (attempt_dir / "render_stderr.txt").write_text(msg)
        return {
            "success": False,
            "return_code": None,
            "gif_path": None,
            "stderr": msg,
            "missing_joint": "right_shoulder",
            "attempt_dir": str(attempt_dir),
        }

    cmd = [
        PYTHON_EXE,
        str(WORK_DIR / "examples" / "annotations_to_animation.py"),
        str(attempt_dir),
        str(motion_path),
        str(retarget_path),
    ]

    result = subprocess.run(
        cmd,
        env=env,
        cwd=str(WORK_DIR),
        capture_output=True,
        text=True,
    )

    (attempt_dir / "render_stdout.txt").write_text(result.stdout[-10000:])
    (attempt_dir / "render_stderr.txt").write_text(result.stderr[-10000:])

    gif_path = attempt_dir / "video.gif"
    success = gif_path.exists()

    return {
        "success": success,
        "return_code": result.returncode,
        "gif_path": str(gif_path) if success else None,
        "stderr": result.stderr[-2000:],
        "missing_joint": parse_missing_joint(result.stderr),
        "attempt_dir": str(attempt_dir),
    }

def build_attempts(character, requested_motion, requested_retarget):
    requested_name = Path(requested_motion).stem

    attempts = [{
        "character": character,
        "motion": requested_motion,
        "retarget": requested_retarget,
        "reason": "requested",
    }]

    motion_specific_retargets = {
        "jumping_jacks": [RETARGET["cmu1_pfp"], RETARGET["fair1_ppf"]],
        "jesse_dance": [RETARGET["mixamo_fff"], RETARGET["fair1_ppf"]],
        "zombie": [RETARGET["fair1_ppf"], RETARGET["mixamo_fff"]],
    }

    for retarget in motion_specific_retargets.get(requested_name, []):
        attempts.append({
            "character": character,
            "motion": requested_motion,
            "retarget": retarget,
            "reason": f"same_character_alt_retarget:{Path(retarget).stem}",
        })

    for safe_char in [CHAR["char1"], CHAR["char2"], CHAR["char3"], CHAR["char4"]]:
        attempts.append({
            "character": safe_char,
            "motion": requested_motion,
            "retarget": requested_retarget,
            "reason": f"safe_character_same_motion:{Path(safe_char).name}",
        })

    for safe_char, safe_motion in [
        (CHAR["char1"], MOTION["wave_hello"]),
        (CHAR["char2"], MOTION["dab"]),
    ]:
        attempts.append({
            "character": safe_char,
            "motion": safe_motion,
            "retarget": RETARGET["fair1_ppf"],
            "reason": f"last_resort:{Path(safe_char).name}+{Path(safe_motion).stem}",
        })

    deduped = []
    seen = set()
    for a in attempts:
        key = (a["character"], a["motion"], a["retarget"])
        if key not in seen:
            seen.add(key)
            deduped.append(a)

    return deduped

def render_with_fallback(index, character, requested_motion, requested_retarget):
    action = Path(requested_motion).stem
    attempts = build_attempts(character, requested_motion, requested_retarget)
    all_attempts = []

    for attempt_no, attempt in enumerate(attempts, 1):
        safe_reason = attempt["reason"].replace(":", "_").replace("+", "_")
        attempt_dir = ROOT_OUTPUT / f"{index:02d}_{action}" / f"attempt_{attempt_no:02d}_{safe_reason}"

        print(
            f"   ↳ attempt {attempt_no}: "
            f"{Path(attempt['character']).name} + "
            f"{Path(attempt['motion']).stem} + "
            f"{Path(attempt['retarget']).stem}"
        )

        r = render_once(
            attempt["character"],
            attempt["motion"],
            attempt["retarget"],
            attempt_dir,
        )

        record = {
            **attempt,
            **r,
            "attempt_no": attempt_no,
        }
        all_attempts.append(record)

        if r["success"]:
            return {
                "index": index,
                "requested_action": action,
                "action": Path(attempt["motion"]).stem,
                "character": Path(attempt["character"]).name,
                "motion": attempt["motion"],
                "retarget": attempt["retarget"],
                "gif_path": r["gif_path"],
                "success": True,
                "fallback_used": attempt_no > 1,
                "fallback_reason": attempt["reason"],
                "attempts": all_attempts,
                "error": "",
                "missing_joint": None,
            }

    last = all_attempts[-1]
    return {
        "index": index,
        "requested_action": action,
        "action": action,
        "character": Path(character).name,
        "motion": requested_motion,
        "retarget": requested_retarget,
        "gif_path": None,
        "success": False,
        "fallback_used": True,
        "fallback_reason": "all_attempts_failed",
        "attempts": all_attempts,
        "error": last.get("stderr", "")[:1000],
        "missing_joint": last.get("missing_joint"),
    }

def make_sequential_gif(items, output_path):
    valid = [item for item in items if item.get("gif_path") and Path(item["gif_path"]).exists()]
    if not valid:
        return None

    resampling = getattr(Image, "Resampling", Image).LANCZOS

    canvas_w = 560
    canvas_h = 560
    label_h = 48
    max_frames_per_clip = 220

    output_frames = []
    durations = []

    for item in valid:
        gif = Image.open(item["gif_path"])

        label = f"{item['index']}. {item['requested_action']}"
        if item.get("fallback_used"):
            label += f" -> {item['action']}"

        frames = list(ImageSequence.Iterator(gif))[:max_frames_per_clip]

        for frame in frames:
            duration = frame.info.get("duration", gif.info.get("duration", 80))

            fr = frame.convert("RGBA")
            fr.thumbnail((canvas_w, canvas_h - label_h), resampling)

            canvas_rgba = Image.new("RGBA", (canvas_w, canvas_h), "white")
            draw = ImageDraw.Draw(canvas_rgba)

            x = (canvas_w - fr.width) // 2
            y = label_h + ((canvas_h - label_h - fr.height) // 2)

            canvas_rgba.alpha_composite(fr, (x, y))
            draw.text((14, 16), label, fill="black")

            output_frames.append(canvas_rgba.convert("RGB"))
            durations.append(duration)

        if output_frames:
            output_frames.append(output_frames[-1].copy())
            durations.append(600)

    output_frames[0].save(
        output_path,
        save_all=True,
        append_images=output_frames[1:],
        duration=durations,
        loop=0,
        optimize=False,
    )

    return str(output_path)

jobs = [
    (CHAR["char1"], MOTION["dab"], RETARGET["fair1_ppf"]),
    (CHAR["char2"], MOTION["wave_hello"], RETARGET["fair1_ppf"]),
    (CHAR["char3"], MOTION["jumping"], RETARGET["fair1_ppf"]),
    (CHAR["char4"], MOTION["jumping_jacks"], RETARGET["cmu1_pfp"]),
    (CHAR["char5"], MOTION["zombie"], RETARGET["fair1_ppf"]),
    (CHAR["char6"], MOTION["jesse_dance"], RETARGET["mixamo_fff"]),
]

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)
else:
    mlflow.set_tracking_uri("file:///kaggle/working/mlruns")
    print("⚠️ MLFLOW_TRACKING_URI가 없어 Kaggle 로컬 mlruns에 기록합니다.")

mlflow.set_experiment("animated-drawings-six-actions-fallback")

started = time.time()
results = []

with mlflow.start_run(run_name="six-actions-render-with-fallback-sequential") as run:
    for index, (character, motion, retarget) in enumerate(jobs, 1):
        print(f"▶️ {index}/6 요청: {Path(character).name} + {Path(motion).stem}")

        item = render_with_fallback(index, character, motion, retarget)
        results.append(item)

        mlflow.log_param(f"requested_action_{index}", item["requested_action"])
        mlflow.log_param(f"actual_action_{index}", item["action"])
        mlflow.log_param(f"fallback_used_{index}", item["fallback_used"])
        mlflow.log_param(f"fallback_reason_{index}", item["fallback_reason"])
        mlflow.log_metric(f"success_{index}", 1 if item["success"] else 0)
        mlflow.log_metric(f"attempt_count_{index}", len(item["attempts"]))

        print(
            f"   result: success={item['success']} "
            f"fallback={item['fallback_used']} "
            f"gif={bool(item['gif_path'])}"
        )

    combined_path = ROOT_OUTPUT / "combined_sequential.gif"
    combined = make_sequential_gif(results, combined_path)

    summary_path = ROOT_OUTPUT / "render_summary.json"
    summary_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    mlflow.log_metric("success_count", sum(1 for r in results if r["success"]))
    mlflow.log_metric("elapsed_seconds", time.time() - started)

    if combined:
        mlflow.log_artifact(combined, artifact_path="combined_gif")

    mlflow.log_artifacts(str(ROOT_OUTPUT), artifact_path="six_actions_with_fallback")

    print("DagsHub run id:", run.info.run_id)
    print("DagsHub uri:", mlflow.get_tracking_uri())

def gif_to_data_uri(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/gif;base64,{data}"

cells = []
for item in results:
    if item["gif_path"] and os.path.exists(item["gif_path"]):
        tag = "fallback" if item["fallback_used"] else "original"
        cells.append(f"""
        <td style="padding:10px;text-align:center;vertical-align:top">
          <div style="font-weight:600;margin-bottom:6px">
            {item["index"]}. {html.escape(item["requested_action"])} ({tag})
          </div>
          <img src="{gif_to_data_uri(item["gif_path"])}" width="220">
        </td>
        """)
    else:
        cells.append(f"""
        <td style="padding:10px;color:#b00020;vertical-align:top">
          <b>{item["index"]}. {html.escape(item["requested_action"])}</b>
          <pre>{html.escape(item["error"][:500])}</pre>
        </td>
        """)

rows = ["<tr>" + "".join(cells[i:i+3]) + "</tr>" for i in range(0, len(cells), 3)]
display(HTML("<table>" + "".join(rows) + "</table>"))

if combined and os.path.exists(combined):
    print("🎬 순차 재생 합본 GIF")
    print(combined)
    display(IPyImage(filename=combined))
else:
    print("❌ 순차 합본 GIF 생성 실패")

In [ ]:
#  에니메이션 최종본 경량버전
import os, time, subprocess, shutil, json, re
from pathlib import Path

import mlflow
from IPython.display import Video, display

WORK_DIR = Path("/kaggle/working/AnimatedDrawings")
PYTHON_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/python"
PIP_EXE = "/kaggle/working/miniconda/envs/ad_env/bin/pip"

ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions_fallback")
ROOT_OUTPUT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [PIP_EXE, "install", "-q", "tqdm", "pillow", "pyyaml", "imageio", "imageio-ffmpeg"],
    check=False,
)

anno = WORK_DIR / "examples" / "annotations_to_animation.py"
src = anno.read_text()
if "'view': {'USE_MESA': True" not in src and '"view": {"USE_MESA": True' not in src:
    src = src.replace(
        "'controller': {",
        "'view': {'USE_MESA': True},\n        'controller': {",
        1,
    )
    anno.write_text(src)
    print("✅ USE_MESA patch 완료")

def p(*parts):
    return str(WORK_DIR.joinpath(*parts))

CHAR = {
    "char1": p("examples", "characters", "char1"),
    "char2": p("examples", "characters", "char2"),
    "char3": p("examples", "characters", "char3"),
    "char4": p("examples", "characters", "char4"),
    "char5": p("examples", "characters", "char5"),
    "char6": p("examples", "characters", "char6"),
}

MOTION = {
    "dab": p("examples", "config", "motion", "dab.yaml"),
    "wave_hello": p("examples", "config", "motion", "wave_hello.yaml"),
    "jumping": p("examples", "config", "motion", "jumping.yaml"),
    "jumping_jacks": p("examples", "config", "motion", "jumping_jacks.yaml"),
    "zombie": p("examples", "config", "motion", "zombie.yaml"),
    "jesse_dance": p("examples", "config", "motion", "jesse_dance.yaml"),
}

RETARGET = {
    "fair1_ppf": p("examples", "config", "retarget", "fair1_ppf.yaml"),
    "cmu1_pfp": p("examples", "config", "retarget", "cmu1_pfp.yaml"),
    "mixamo_fff": p("examples", "config", "retarget", "mixamo_fff.yaml"),
}

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = str(WORK_DIR)

def char_has_joint(char_dir, joint_name):
    cfg = Path(char_dir) / "char_cfg.yaml"
    if not cfg.exists():
        return False
    return joint_name.lower() in cfg.read_text(errors="ignore").lower()

def parse_missing_joint(stderr):
    m = re.search(r"Could not find joint1 in runtime check:\s*([A-Za-z0-9_]+)", stderr)
    return m.group(1) if m else None

def copy_character(src_char_dir, dst_dir):
    dst = Path(dst_dir)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_char_dir, dst)
    return dst

def render_once(src_char_dir, motion_path, retarget_path, attempt_dir):
    attempt_dir = copy_character(src_char_dir, attempt_dir)

    if not char_has_joint(attempt_dir, "right_shoulder"):
        msg = "precheck_missing_joint:right_shoulder"
        (attempt_dir / "render_stderr.txt").write_text(msg)
        return {
            "success": False,
            "return_code": None,
            "gif_path": None,
            "stderr": msg,
            "missing_joint": "right_shoulder",
            "attempt_dir": str(attempt_dir),
        }

    cmd = [
        PYTHON_EXE,
        str(WORK_DIR / "examples" / "annotations_to_animation.py"),
        str(attempt_dir),
        str(motion_path),
        str(retarget_path),
    ]

    result = subprocess.run(
        cmd,
        env=env,
        cwd=str(WORK_DIR),
        capture_output=True,
        text=True,
    )

    (attempt_dir / "render_stdout.txt").write_text(result.stdout[-10000:])
    (attempt_dir / "render_stderr.txt").write_text(result.stderr[-10000:])

    gif_path = attempt_dir / "video.gif"
    success = gif_path.exists()

    return {
        "success": success,
        "return_code": result.returncode,
        "gif_path": str(gif_path) if success else None,
        "stderr": result.stderr[-2000:],
        "missing_joint": parse_missing_joint(result.stderr),
        "attempt_dir": str(attempt_dir),
    }

def build_attempts(character, requested_motion, requested_retarget):
    requested_name = Path(requested_motion).stem

    attempts = [{
        "character": character,
        "motion": requested_motion,
        "retarget": requested_retarget,
        "reason": "requested",
    }]

    motion_specific_retargets = {
        "jumping_jacks": [RETARGET["cmu1_pfp"], RETARGET["fair1_ppf"]],
        "jesse_dance": [RETARGET["mixamo_fff"], RETARGET["fair1_ppf"]],
        "zombie": [RETARGET["fair1_ppf"], RETARGET["mixamo_fff"]],
    }

    for retarget in motion_specific_retargets.get(requested_name, []):
        attempts.append({
            "character": character,
            "motion": requested_motion,
            "retarget": retarget,
            "reason": f"same_character_alt_retarget:{Path(retarget).stem}",
        })

    for safe_char in [CHAR["char1"], CHAR["char2"], CHAR["char3"], CHAR["char4"]]:
        attempts.append({
            "character": safe_char,
            "motion": requested_motion,
            "retarget": requested_retarget,
            "reason": f"safe_character_same_motion:{Path(safe_char).name}",
        })

    for safe_char, safe_motion in [
        (CHAR["char1"], MOTION["wave_hello"]),
        (CHAR["char2"], MOTION["dab"]),
    ]:
        attempts.append({
            "character": safe_char,
            "motion": safe_motion,
            "retarget": RETARGET["fair1_ppf"],
            "reason": f"last_resort:{Path(safe_char).name}+{Path(safe_motion).stem}",
        })

    deduped = []
    seen = set()
    for a in attempts:
        key = (a["character"], a["motion"], a["retarget"])
        if key not in seen:
            seen.add(key)
            deduped.append(a)

    return deduped

def render_with_fallback(index, character, requested_motion, requested_retarget):
    action = Path(requested_motion).stem
    attempts = build_attempts(character, requested_motion, requested_retarget)
    all_attempts = []

    for attempt_no, attempt in enumerate(attempts, 1):
        safe_reason = attempt["reason"].replace(":", "_").replace("+", "_")
        attempt_dir = ROOT_OUTPUT / f"{index:02d}_{action}" / f"attempt_{attempt_no:02d}_{safe_reason}"

        print(
            f"   ↳ attempt {attempt_no}: "
            f"{Path(attempt['character']).name} + "
            f"{Path(attempt['motion']).stem} + "
            f"{Path(attempt['retarget']).stem}"
        )

        r = render_once(
            attempt["character"],
            attempt["motion"],
            attempt["retarget"],
            attempt_dir,
        )

        record = {**attempt, **r, "attempt_no": attempt_no}
        all_attempts.append(record)

        if r["success"]:
            return {
                "index": index,
                "requested_action": action,
                "action": Path(attempt["motion"]).stem,
                "character": Path(attempt["character"]).name,
                "motion": attempt["motion"],
                "retarget": attempt["retarget"],
                "gif_path": r["gif_path"],
                "success": True,
                "fallback_used": attempt_no > 1,
                "fallback_reason": attempt["reason"],
                "attempts": all_attempts,
                "error": "",
                "missing_joint": None,
            }

    last = all_attempts[-1]
    return {
        "index": index,
        "requested_action": action,
        "action": action,
        "character": Path(character).name,
        "motion": requested_motion,
        "retarget": requested_retarget,
        "gif_path": None,
        "success": False,
        "fallback_used": True,
        "fallback_reason": "all_attempts_failed",
        "attempts": all_attempts,
        "error": last.get("stderr", "")[:1000],
        "missing_joint": last.get("missing_joint"),
    }

def make_sequential_mp4(items, output_path):
    valid = [item for item in items if item.get("gif_path") and Path(item["gif_path"]).exists()]
    if not valid:
        return None

    segments_dir = ROOT_OUTPUT / "mp4_segments"
    if segments_dir.exists():
        shutil.rmtree(segments_dir)
    segments_dir.mkdir(parents=True, exist_ok=True)

    segment_paths = []

    for item in valid:
        label = f"{item['index']}_{item['requested_action']}"
        if item.get("fallback_used"):
            label += f"_fallback_{item['action']}"

        segment_path = segments_dir / f"{item['index']:02d}_{item['requested_action']}.mp4"

        vf = (
            "fps=12,"
            "scale=512:512:force_original_aspect_ratio=decrease,"
            "pad=512:512:(ow-iw)/2:(oh-ih)/2:color=white,"
            "format=yuv420p"
        )

        cmd = [
            "ffmpeg", "-y",
            "-i", item["gif_path"],
            "-vf", vf,
            "-an",
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "23",
            str(segment_path),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0 and segment_path.exists():
            segment_paths.append(segment_path)
            print(f"✅ MP4 segment 생성: {segment_path}")
        else:
            print(f"❌ segment 실패: {item['gif_path']}")
            print(result.stderr[-800:])

    if not segment_paths:
        return None

    concat_file = segments_dir / "concat.txt"
    concat_file.write_text(
        "\n".join([f"file '{path.as_posix()}'" for path in segment_paths]),
        encoding="utf-8",
    )

    cmd_concat = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(concat_file),
        "-c", "copy",
        str(output_path),
    ]

    result = subprocess.run(cmd_concat, capture_output=True, text=True)
    if result.returncode != 0:
        print("❌ concat copy 실패. 재인코딩으로 재시도합니다.")
        print(result.stderr[-800:])

        cmd_concat_reencode = [
            "ffmpeg", "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", str(concat_file),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", "veryfast",
            "-crf", "23",
            str(output_path),
        ]
        result = subprocess.run(cmd_concat_reencode, capture_output=True, text=True)

    if output_path.exists():
        return str(output_path)

    print("❌ 순차 MP4 생성 실패")
    print(result.stderr[-1000:])
    return None

jobs = [
    (CHAR["char1"], MOTION["dab"], RETARGET["fair1_ppf"]),
    (CHAR["char2"], MOTION["wave_hello"], RETARGET["fair1_ppf"]),
    (CHAR["char3"], MOTION["jumping"], RETARGET["fair1_ppf"]),
    (CHAR["char4"], MOTION["jumping_jacks"], RETARGET["cmu1_pfp"]),
    (CHAR["char5"], MOTION["zombie"], RETARGET["fair1_ppf"]),
    (CHAR["char6"], MOTION["jesse_dance"], RETARGET["mixamo_fff"]),
]

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)
else:
    mlflow.set_tracking_uri("file:///kaggle/working/mlruns")
    print("⚠️ MLFLOW_TRACKING_URI가 없어 Kaggle 로컬 mlruns에 기록합니다.")

mlflow.set_experiment("animated-drawings-six-actions-fallback")

started = time.time()
results = []

with mlflow.start_run(run_name="six-actions-render-with-fallback-sequential-mp4") as run:
    for index, (character, motion, retarget) in enumerate(jobs, 1):
        print(f"▶️ {index}/6 요청: {Path(character).name} + {Path(motion).stem}")

        item = render_with_fallback(index, character, motion, retarget)
        results.append(item)

        mlflow.log_param(f"requested_action_{index}", item["requested_action"])
        mlflow.log_param(f"actual_action_{index}", item["action"])
        mlflow.log_param(f"fallback_used_{index}", item["fallback_used"])
        mlflow.log_param(f"fallback_reason_{index}", item["fallback_reason"])
        mlflow.log_metric(f"success_{index}", 1 if item["success"] else 0)
        mlflow.log_metric(f"attempt_count_{index}", len(item["attempts"]))

        print(
            f"   result: success={item['success']} "
            f"fallback={item['fallback_used']} "
            f"gif={item['gif_path']}"
        )

    summary_path = ROOT_OUTPUT / "render_summary.json"
    summary_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    combined_mp4 = make_sequential_mp4(
        results,
        ROOT_OUTPUT / "combined_sequential.mp4",
    )

    mlflow.log_metric("success_count", sum(1 for r in results if r["success"]))
    mlflow.log_metric("elapsed_seconds", time.time() - started)

    mlflow.log_artifact(str(summary_path), artifact_path="summary")

    for item in results:
        if item.get("gif_path") and Path(item["gif_path"]).exists():
            mlflow.log_artifact(item["gif_path"], artifact_path="individual_gifs")

    if combined_mp4:
        mlflow.log_artifact(combined_mp4, artifact_path="combined_video")

    print("DagsHub run id:", run.info.run_id)
    print("DagsHub uri:", mlflow.get_tracking_uri())

print("\n개별 GIF 경로:")
for item in results:
    print(
        f"{item['index']}. requested={item['requested_action']} "
        f"actual={item['action']} "
        f"fallback={item['fallback_used']} "
        f"path={item['gif_path']}"
    )

if combined_mp4 and Path(combined_mp4).exists():
    print("\n🎬 순차 재생 합본 MP4:")
    print(combined_mp4)
    display(Video(combined_mp4, embed=True, width=520))
else:
    print("❌ 순차 MP4 생성 실패")

TTS 만들기

In [ ]:
!pip install -q --no-cache-dir torchaudio --index-url https://download.pytorch.org/whl/cu130

GPT-SoVITS -> torchaudio.load -> soundfile.read -> 정상 wav 로딩


In [ ]:
# 중요: 이 셀 실행 전에는 torchvision을 import하지 마세요.
# 출력 목표: /kaggle/working/final_tts_Vlog_Opening_Excited.wav

import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_IMAGE_PROCESSING"] = "1"

import sys
import subprocess
import pkgutil
import importlib.machinery
from pathlib import Path

import torch
import soundfile as sf
from gtts import gTTS
from huggingface_hub import snapshot_download
from IPython.display import Audio, display

# Python 3.12 compatibility patch
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})

if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def _find_module(self, fullname, path=None):
        spec = self.find_spec(fullname)
        return spec.loader if spec else None
    importlib.machinery.FileFinder.find_module = _find_module

WORK = Path("/kaggle/working")
GPT_DIR = WORK / "GPT-SoVITS"
PRETRAINED = GPT_DIR / "GPT_SoVITS" / "pretrained_models"
TTS_OUT = WORK / "final_tts_Vlog_Opening_Excited.wav"

os.chdir(WORK)

if not GPT_DIR.exists():
    print("▶️ GPT-SoVITS 소스 다운로드...")
    subprocess.run(
        ["git", "clone", "https://github.com/RVC-Boss/GPT-SoVITS.git", str(GPT_DIR)],
        check=True,
    )

PRETRAINED.mkdir(parents=True, exist_ok=True)

if not (PRETRAINED / "chinese-roberta-wwm-ext-large").exists():
    print("▶️ GPT-SoVITS 사전 모델 다운로드/복구...")
    snapshot_download(
        repo_id="lj1995/GPT-SoVITS",
        local_dir=str(PRETRAINED),
        local_dir_use_symlinks=False,
    )

os.chdir(GPT_DIR)

for p in [str(GPT_DIR), str(GPT_DIR / "GPT_SoVITS")]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from BigVGAN import bigvgan
except ImportError:
    import bigvgan

if hasattr(bigvgan.BigVGAN, "_from_pretrained") and not getattr(bigvgan.BigVGAN, "_is_patched", False):
    orig_from_pretrained = bigvgan.BigVGAN._from_pretrained.__func__

    @classmethod
    def patched_from_pretrained(cls, *args, **kwargs):
        kwargs.setdefault("proxies", None)
        kwargs.setdefault("resume_download", False)
        return orig_from_pretrained(cls, *args, **kwargs)

    bigvgan.BigVGAN._from_pretrained = patched_from_pretrained
    bigvgan.BigVGAN._is_patched = True

import inference_webui
from inference_webui import change_gpt_weights, change_sovits_weights, get_tts_wav, dict_language

# ✅ torchcodec 우회: inference_webui 내부 torchaudio까지 soundfile 기반으로 강제 교체
import torchaudio

def safe_torchaudio_load(filename, *args, **kwargs):
    audio, sr = sf.read(filename, always_2d=True)
    audio = torch.from_numpy(audio.T).float()
    return audio, sr

torchaudio.load = safe_torchaudio_load
inference_webui.torchaudio.load = safe_torchaudio_load

print("✅ torchaudio.load patched:", torchaudio.load)
print("✅ inference_webui.torchaudio.load patched:", inference_webui.torchaudio.load)

dict_language["ko"] = "all_ko"
dict_language["한국어"] = "all_ko"

class DummyData:
    sampling_rate = 32000

class DummyHPS:
    data = DummyData()

inference_webui.hps = DummyHPS()

import text.korean
from mecab import MeCab
import mecab_ko_dic

try:
    dic_path = mecab_ko_dic.dictionary_path
except AttributeError:
    dic_path = "/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dicdir"

text.korean._g2p.mecab = MeCab(dictionary_path=dic_path)

print("▶️ GPT-SoVITS V3 가중치 로딩 중...")

gpt_loader = change_gpt_weights("GPT_SoVITS/pretrained_models/s1v3.ckpt")
if hasattr(gpt_loader, "__iter__") and not isinstance(gpt_loader, str):
    for _ in gpt_loader:
        pass

sovits_loader = change_sovits_weights("GPT_SoVITS/pretrained_models/s2Gv3.pth", "ko", "ko")
if hasattr(sovits_loader, "__iter__") and not isinstance(sovits_loader, str):
    for _ in sovits_loader:
        pass

REF_WAV = GPT_DIR / "ref_v6_fixed.wav"
medium_prompt = "안녕하세요. 지금부터 인공지능 음성 합성 테스트를 시작하겠습니다."

if not REF_WAV.exists():
    print("▶️ 기준 음성 ref_wav 생성 중...")
    ref_mp3 = GPT_DIR / "ref_temp_v6.mp3"
    gTTS(medium_prompt, lang="ko").save(str(ref_mp3))
    subprocess.run([
        "ffmpeg", "-y",
        "-i", str(ref_mp3),
        "-ar", "16000",
        "-ac", "1",
        str(REF_WAV),
        "-loglevel", "quiet",
    ], check=True)

text_to_speak = (
    "우와, 드디어 기다리고 기다리던 전주 한옥마을에 도착했습니다! "
    "길거리 음식 냄새가 벌써부터 너무 좋은데요? 얼른 먹으러 가볼까요?"
)

# ✅ 혹시 중간 import로 덮였을 가능성을 막기 위해 생성 직전 한 번 더 패치
torchaudio.load = safe_torchaudio_load
inference_webui.torchaudio.load = safe_torchaudio_load

print("▶️ TTS 생성 중...")
generator = get_tts_wav(
    ref_wav_path=str(REF_WAV),
    prompt_text=medium_prompt,
    prompt_language="ko",
    text=text_to_speak,
    text_language="ko",
)

sr, audio_data = next(generator)
sf.write(str(TTS_OUT), audio_data, sr)

print("✅ TTS 생성 완료:", TTS_OUT)
print("exists:", TTS_OUT.exists())

display(Audio(str(TTS_OUT)))

AnimatedDrawings 순차 MP4
+ GPT-SoVITS TTS
+ MusicGen BGM
+ FFmpeg 믹싱
= 최종 브이로그 MP4

A. 애니메이션 + MusicGen 로컬 합성

 지금 만든 경량 애니메이션 코드는 1단계: 순차 애니메이션 MP4 생성까지
이후는 다시 렌더링하지 말고, /kaggle/working/six_doodle_actions_fallback/combined_sequential.mp4를 재사용


In [ ]:
# A. 애니메이션 + MusicGen 로컬 합성 최종본

import os, time, subprocess, gc
from pathlib import Path

import torch
import scipy.io.wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from IPython.display import Video, display

ANIMATION_MP4 = Path("/kaggle/working/six_doodle_actions_fallback/combined_sequential.mp4")
BGM_WAV = Path("/kaggle/working/musicgen_travel_bgm.wav")
ANIM_BGM_MP4 = Path("/kaggle/working/animation_with_musicgen.mp4")

assert ANIMATION_MP4.exists(), f"애니메이션 MP4가 없습니다: {ANIMATION_MP4}"

def generate_musicgen_bgm(out_path):
    if out_path.exists():
        print("✅ 기존 BGM 사용:", out_path)
        return out_path

    prompt = "A cheerful acoustic guitar travel vlog instrumental, happy, bright"
    print("▶️ MusicGen BGM 생성 중...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

    inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(device)

    started = time.time()
    audio_values = model.generate(**inputs, max_new_tokens=750)
    print(f"✅ BGM 생성 완료: {time.time() - started:.1f}초")

    sampling_rate = model.config.audio_encoder.sampling_rate
    audio_np = audio_values[0, 0].detach().cpu().numpy()
    scipy.io.wavfile.write(str(out_path), rate=sampling_rate, data=audio_np)

    del model, processor, inputs, audio_values
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out_path

def mux_animation_bgm(animation_mp4, bgm_wav, out_mp4):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(animation_mp4),
        "-stream_loop", "-1",
        "-i", str(bgm_wav),
        "-filter_complex", "[1:a]volume=-10dB[aout]",
        "-map", "0:v",
        "-map", "[aout]",
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(out_mp4),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("FFmpeg 합성 실패")

    return out_mp4

generate_musicgen_bgm(BGM_WAV)
mux_animation_bgm(ANIMATION_MP4, BGM_WAV, ANIM_BGM_MP4)

print("🎬 애니메이션 + MusicGen:", ANIM_BGM_MP4)
display(Video(str(ANIM_BGM_MP4), embed=True, width=640))

B. 애니메이션 + MusicGen + TTS 로컬 합성



In [ ]:
# B. 애니메이션 + MusicGen + TTS 로컬 합성


import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_IMAGE_PROCESSING"] = "1"

import gc
import subprocess
from pathlib import Path

import torch
import scipy.io.wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from IPython.display import Video, Audio, display

ANIMATION_MP4 = Path("/kaggle/working/six_doodle_actions_fallback/combined_sequential.mp4")
TTS_WAV = Path("/kaggle/working/final_tts_Vlog_Opening_Excited.wav")
BGM_WAV = Path("/kaggle/working/musicgen_travel_bgm.wav")
FINAL_MP4 = Path("/kaggle/working/animation_musicgen_tts_final.mp4")

assert ANIMATION_MP4.exists(), f"애니메이션 MP4 없음: {ANIMATION_MP4}"
assert TTS_WAV.exists(), f"TTS 없음: {TTS_WAV}"

print("✅ 애니메이션:", ANIMATION_MP4)
print("✅ TTS:", TTS_WAV)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def generate_musicgen_bgm(out_path):
    if out_path.exists():
        print("✅ 기존 MusicGen BGM 사용:", out_path)
        return out_path

    print("▶️ MusicGen BGM 생성 중...")
    prompt = "A cheerful acoustic guitar travel vlog instrumental, happy, bright"

    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

    inputs = processor(
        text=[prompt],
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        audio_values = model.generate(
            **inputs,
            max_new_tokens=750,  # 약 15초
        )

    sampling_rate = model.config.audio_encoder.sampling_rate
    audio_np = audio_values[0, 0].detach().cpu().numpy()

    scipy.io.wavfile.write(str(out_path), rate=sampling_rate, data=audio_np)

    del model, processor, inputs, audio_values
    free_memory()

    print("✅ MusicGen BGM 생성 완료:", out_path)
    return out_path

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    return float(result.stdout.strip())

def mix_animation_tts_bgm(animation_mp4, tts_wav, bgm_wav, out_mp4):
    video_duration = get_duration(animation_mp4)
    print("🎬 animation duration:", video_duration)

    filter_complex = (
        f"[1:a]volume=1.4,apad,atrim=0:{video_duration}[tts];"
        f"[2:a]volume=-10dB,atrim=0:{video_duration}[bgm];"
        "[tts][bgm]amix=inputs=2:duration=longest:dropout_transition=0:normalize=0[aout]"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", str(animation_mp4),
        "-i", str(tts_wav),
        "-stream_loop", "-1",
        "-i", str(bgm_wav),
        "-filter_complex", filter_complex,
        "-map", "0:v",
        "-map", "[aout]",
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(out_mp4),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print("❌ FFmpeg 실패:")
        print(result.stderr[-2000:])
        raise RuntimeError("FFmpeg 합성 실패")

    print("✅ 최종 합성 완료:", out_mp4)
    return out_mp4

generate_musicgen_bgm(BGM_WAV)

print("🎧 TTS 미리듣기")
display(Audio(str(TTS_WAV)))

FINAL_MP4 = mix_animation_tts_bgm(
    ANIMATION_MP4,
    TTS_WAV,
    BGM_WAV,
    FINAL_MP4,
)

print("🎬 최종 영상:", FINAL_MP4)
display(Video(str(FINAL_MP4), embed=True, width=640))

In [ ]:
!pip install -q --no-cache-dir --force-reinstall \
  "huggingface_hub==0.23.5" \
  "transformers==4.46.3" \
  "tokenizers==0.20.3" \
  "peft==0.11.1" \
  "accelerate==0.33.0" \
  "click==8.1.7" \
  "typer==0.12.5" \
  "fsspec==2025.3.0" \
  "numpy==1.26.4" \
  "pandas==2.1.4"

In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
import torch, huggingface_hub, transformers, tokenizers, accelerate, click, fsspec, numpy, pandas

print("torch:", torch.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)
print("click:", click.__version__)
print("fsspec:", fsspec.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

In [ ]:
!pip install -q --no-cache-dir --force-reinstall torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_IMAGE_PROCESSING"] = "1" 

In [ ]:
!pip install -q --no-cache-dir --force-reinstall \
  "transformers==4.45.2" \
  "tokenizers==0.20.3" \
  "huggingface_hub==0.23.5" \
  "peft==0.11.1" \
  "accelerate==0.33.0" \
  "click==8.1.7" \
  "typer==0.12.5" \
  "fsspec==2025.3.0" \
  "numpy==1.26.4" \
  "pandas==2.1.4"

In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
import torch, torchvision, torchaudio
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
import peft
print("peft:", peft.__version__)

In [ ]:
# 멀티 시나리오 tts


import time
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

WORK = Path("/kaggle/working")

tts_scenarios = [
    {
        "name": "01_vlog_opening_excited",
        "text": "우와, 드디어 기다리고 기다리던 전주 한옥마을에 도착했습니다! 길거리 음식 냄새가 벌써부터 너무 좋은데요? 얼른 먹으러 가볼까요?",
    },
    {
        "name": "02_place_guide_informative",
        "text": "이곳은 오래된 한옥의 아름다움을 가까이에서 느낄 수 있는 대표적인 여행지입니다. 골목마다 사진 찍기 좋은 장소가 숨어 있으니 천천히 둘러보세요.",
    },
    {
        "name": "03_food_review_delighted",
        "text": "방금 먹어본 전주비빔밥은 정말 고소하고 풍성했어요. 여러 가지 나물이 한 그릇 안에서 조화롭게 어우러져서 계속 생각나는 맛입니다.",
    },
    {
        "name": "04_calm_healing_walk",
        "text": "잠시 걸음을 늦추고 주변 풍경을 바라봅니다. 따뜻한 햇살과 조용한 골목길 덕분에 마음이 한결 편안해지는 것 같아요.",
    },
    {
        "name": "05_vlog_outro_relaxing",
        "text": "오늘 여행은 여기까지입니다. 짧은 시간이었지만 맛있는 음식과 아름다운 풍경을 가득 담을 수 있어서 정말 행복했습니다. 다음 여행에서 또 만나요.",
    },
]

tts_outputs = []

for scenario in tts_scenarios:
    output_path = WORK / f"final_tts_{scenario['name']}.wav"

    print(f"▶️ TTS 생성 중: {scenario['name']}")
    started = time.time()

    generator = get_tts_wav(
        ref_wav_path=str(REF_WAV),
        prompt_text=medium_prompt,
        prompt_language="ko",
        text=scenario["text"],
        text_language="ko",
    )

    sr, audio_data = next(generator)
    sf.write(str(output_path), audio_data, sr)

    elapsed = time.time() - started
    tts_outputs.append(output_path)

    print(f"✅ 완료: {output_path} ({elapsed:.2f}초)")
    display(Audio(str(output_path)))

print("\n생성된 TTS 파일:")
for path in tts_outputs:
    print(path)

AnimatedDrawings MP4 + GPT-SoVITS TTS + MusicGen BGM + FFmpeg 믹싱 + DagsHub + zrok/FastAPI 

!pip install -qU --no-build-isolation git+https://github.com/facebookresearch/audiocraft.git
대신 transformers의 MusicgenForConditionalGeneration을 쓰는 방식이라 audiocraft 사용 중 (설치시간) 

In [ ]:
# ===== 0) 시스템 패키지 =====
!apt-get update -y
!apt-get install -y ffmpeg git curl mecab libmecab-dev mecab-ipadic-utf8 --quiet

# ===== 1) 공통 기록/서버/미디어 =====
!pip install -q --no-cache-dir \
  mlflow dagshub \
  fastapi uvicorn \
  pillow imageio imageio-ffmpeg \
  soundfile scipy \
  requests gTTS

# ===== 2) MusicGen / HuggingFace =====
!pip install -q --no-cache-dir \
  transformers accelerate \
  "huggingface_hub<0.24.0"

# ===== 3) GPT-SoVITS 한국어/TTS 의존성 =====
!pip install -q --no-cache-dir \
  gradio typer click \
  split_lang x_transformers pytorch_lightning \
  cn2an pypinyin jieba_fast \
  mecab-python3 python-mecab-ko mecab-ko-dic \
  g2pk2 ko_pron fast_langdetect wordsegment g2p_en konlpy

# ===== 4) chumpy 빌드 안정화 =====
!pip install -q --no-cache-dir setuptools==65.5.0 wheel six
!pip install -q --no-cache-dir chumpy --no-build-isolation

# ===== 5) numpy/pandas/fsspec 최종 고정 =====
# 중요: 모든 설치가 끝난 뒤 마지막에 실행
!pip install -q --no-cache-dir --force-reinstall \
  numpy==1.26.4 pandas==2.1.4 fsspec==2025.3.0

In [ ]:
!pip install -q --no-cache-dir --force-reinstall \
  "peft==0.11.1" \
  "accelerate==0.33.0" \
  "huggingface_hub==0.23.5" \
  "transformers==4.46.3" \
  "tokenizers==0.20.3" \
  "click==8.1.7" \
  "typer==0.12.5"

In [ ]:
import peft, accelerate, huggingface_hub, transformers, click
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("click:", click.__version__)

In [ ]:
!pip install -q --no-cache-dir --force-reinstall "accelerate==0.33.0"


In [ ]:
!pip install -q --no-cache-dir --force-reinstall setuptools==65.5.0 wheel six
!pip install -q --no-cache-dir chumpy --no-build-isolation --no-deps

설치 후에는 꼭 Restart Session 하세요. 지금처럼 numpy, pandas, huggingface_hub, click을 바꾼 뒤 재시작 없이 바로 모델을 import하면 런타임이 꼬일 수 있습니다.







In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_IMAGE_PROCESSING"] = "1"

import sys, time, subprocess, shutil, pkgutil, importlib.machinery
from pathlib import Path

import torch
import soundfile as sf
from gtts import gTTS

# Python 3.12 compatibility patch
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})

if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def _find_module(self, fullname, path=None):
        spec = self.find_spec(fullname)
        return spec.loader if spec else None
    importlib.machinery.FileFinder.find_module = _find_module

TTS_ABS = Path("/kaggle/working/final_tts_Vlog_Opening_Excited.wav")
TTS_OLD = Path("/kaggle/working/GPT-SoVITS/final_tts_Vlog_Opening_Excited.wav")
GPT_DIR = Path("/kaggle/working/GPT-SoVITS")

if TTS_OLD.exists() and not TTS_ABS.exists():
    shutil.copy(TTS_OLD, TTS_ABS)
    print("✅ 기존 TTS를 /kaggle/working 으로 복사:", TTS_ABS)

if not TTS_ABS.exists():
    print("▶️ TTS가 없어 GPT-SoVITS로 다시 생성합니다.")

    os.chdir(GPT_DIR)

    for p in [str(GPT_DIR), str(GPT_DIR / "GPT_SoVITS")]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        from BigVGAN import bigvgan
    except ImportError:
        import bigvgan

    if not getattr(bigvgan.BigVGAN, "_is_patched", False):
        if hasattr(bigvgan.BigVGAN, "_from_pretrained"):
            orig_from_pretrained = bigvgan.BigVGAN._from_pretrained.__func__

            @classmethod
            def patched_from_pretrained(cls, *args, **kwargs):
                kwargs.setdefault("proxies", None)
                kwargs.setdefault("resume_download", False)
                return orig_from_pretrained(cls, *args, **kwargs)

            bigvgan.BigVGAN._from_pretrained = patched_from_pretrained
            bigvgan.BigVGAN._is_patched = True

    import inference_webui
    from inference_webui import change_gpt_weights, change_sovits_weights, get_tts_wav, dict_language

    # torchcodec 우회 패치
    import torchaudio

    def safe_torchaudio_load(filename, *args, **kwargs):
        audio, sr = sf.read(filename, always_2d=True)
        audio = torch.from_numpy(audio.T).float()
        return audio, sr

    torchaudio.load = safe_torchaudio_load
    inference_webui.torchaudio.load = safe_torchaudio_load

    dict_language["ko"] = "all_ko"
    dict_language["한국어"] = "all_ko"

    class DummyData:
        sampling_rate = 32000

    class DummyHPS:
        data = DummyData()

    inference_webui.hps = DummyHPS()

    import text.korean
    from mecab import MeCab
    import mecab_ko_dic

    try:
        dic_path = mecab_ko_dic.dictionary_path
    except AttributeError:
        dic_path = "/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dicdir"

    text.korean._g2p.mecab = MeCab(dictionary_path=dic_path)

    gpt_loader = change_gpt_weights("GPT_SoVITS/pretrained_models/s1v3.ckpt")
    if hasattr(gpt_loader, "__iter__") and not isinstance(gpt_loader, str):
        for _ in gpt_loader:
            pass

    sovits_loader = change_sovits_weights("GPT_SoVITS/pretrained_models/s2Gv3.pth", "ko", "ko")
    if hasattr(sovits_loader, "__iter__") and not isinstance(sovits_loader, str):
        for _ in sovits_loader:
            pass

    REF_WAV = GPT_DIR / "ref_v6_fixed.wav"
    medium_prompt = "안녕하세요. 지금부터 인공지능 음성 합성 테스트를 시작하겠습니다."

    if not REF_WAV.exists():
        gTTS(medium_prompt, lang="ko").save(str(GPT_DIR / "ref_temp_v6.mp3"))
        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(GPT_DIR / "ref_temp_v6.mp3"),
            "-ar", "16000",
            "-ac", "1",
            str(REF_WAV),
            "-loglevel", "quiet",
        ], check=True)

    text_to_speak = "우와, 드디어 기다리고 기다리던 전주 한옥마을에 도착했습니다! 길거리 음식 냄새가 벌써부터 너무 좋은데요? 얼른 먹으러 가볼까요?"

    # 생성 직전 한 번 더 패치
    torchaudio.load = safe_torchaudio_load
    inference_webui.torchaudio.load = safe_torchaudio_load

    generator = get_tts_wav(
        ref_wav_path=str(REF_WAV),
        prompt_text=medium_prompt,
        prompt_language="ko",
        text=text_to_speak,
        text_language="ko",
    )

    sr, audio_data = next(generator)
    sf.write(str(TTS_ABS), audio_data, sr)

print("✅ 최종 TTS 경로:", TTS_ABS)
print("exists:", TTS_ABS.exists())

In [ ]:
import os, sys, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

WORK = Path("/kaggle/working")
GPT_DIR = WORK / "GPT-SoVITS"
PRETRAINED = GPT_DIR / "GPT_SoVITS" / "pretrained_models"

os.chdir(WORK)

if not GPT_DIR.exists():
    print("▶️ GPT-SoVITS 소스 다운로드...")
    subprocess.run(
        ["git", "clone", "https://github.com/RVC-Boss/GPT-SoVITS.git", str(GPT_DIR)],
        check=True,
    )

PRETRAINED.mkdir(parents=True, exist_ok=True)

print("▶️ GPT-SoVITS 사전 모델 다운로드/복구...")
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir=str(PRETRAINED),
    local_dir_use_symlinks=False,
)

print("✅ GPT-SoVITS 복구 완료")
print("GPT_DIR exists:", GPT_DIR.exists())
print("PRETRAINED exists:", PRETRAINED.exists())

음성이 한 번만 나오는 게 아니라 각 애니메이션 구간마다 해당 TTS가 같이 재생

In [ ]:
#음성이 한 번만 나오는 게 아니라 각 애니메이션 구간마다 해당 TTS가 같이 재생
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_IMAGE_PROCESSING"] = "1"

import gc, json, time, subprocess
from pathlib import Path

import torch
import soundfile as sf
import scipy.io.wavfile
import mlflow
from IPython.display import Audio, Video, display
from transformers import AutoProcessor, MusicgenForConditionalGeneration

# A 셀에서 준비되어 있어야 함
if "get_tts_wav" not in globals():
    raise RuntimeError("먼저 A 셀을 실행해서 get_tts_wav, REF_WAV, medium_prompt를 준비하세요.")

ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions_fallback")
SUMMARY_PATH = ROOT_OUTPUT / "render_summary.json"
BGM_WAV = Path("/kaggle/working/musicgen_travel_bgm.wav")
FINAL_MP4 = Path("/kaggle/working/animation_musicgen_tts_segment_synced.mp4")

SEGMENT_VIDEO_DIR = ROOT_OUTPUT / "synced_video_segments"
SEGMENT_TTS_DIR = ROOT_OUTPUT / "synced_tts"
SEGMENT_MIX_DIR = ROOT_OUTPUT / "synced_mixed_segments"

for d in [SEGMENT_VIDEO_DIR, SEGMENT_TTS_DIR, SEGMENT_MIX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert SUMMARY_PATH.exists(), f"render_summary.json 없음: {SUMMARY_PATH}"

# 6개 애니메이션에 맞춰 6개 음성. 5개만 쓰려면 마지막 항목을 지워도 됩니다.
tts_scenarios = [
    {
        "name": "01_opening_excited",
        "text": "우와, 드디어 기다리고 기다리던 여행이 시작됐습니다! 첫 번째 캐릭터가 신나게 춤을 추고 있어요.",
    },
    {
        "name": "02_friendly_wave",
        "text": "이번에는 반갑게 손을 흔드는 장면입니다. 여행지에서 만나는 작은 인사처럼 따뜻한 느낌이 나네요.",
    },
    {
        "name": "03_jump_energy",
        "text": "세 번째 장면은 통통 튀는 에너지가 느껴집니다. 발걸음까지 가벼워지는 순간이에요.",
    },
    {
        "name": "04_jumping_jacks",
        "text": "이번 동작은 온몸을 크게 움직이는 활기찬 장면입니다. 여행의 흥이 점점 올라가고 있어요.",
    },
    {
        "name": "05_playful_moment",
        "text": "다섯 번째 장면은 조금 장난스럽고 유쾌한 분위기입니다. 캐릭터마다 다른 매력이 살아납니다.",
    },
    {
        "name": "06_outro_relaxing",
        "text": "마지막 장면입니다. 오늘의 짧은 애니메이션 여행은 여기까지예요. 다음 장면에서 또 만나요.",
    },
]

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path),
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return float(result.stdout.strip())

def ensure_musicgen_bgm(out_path):
    if out_path.exists():
        print("✅ 기존 MusicGen BGM 사용:", out_path)
        return out_path

    print("▶️ MusicGen BGM 생성 중...")
    prompt = "A cheerful acoustic guitar travel vlog instrumental, happy, bright"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

    inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=750)

    sampling_rate = model.config.audio_encoder.sampling_rate
    audio_np = audio_values[0, 0].detach().cpu().numpy()
    scipy.io.wavfile.write(str(out_path), rate=sampling_rate, data=audio_np)

    del model, processor, inputs, audio_values
    free_memory()

    print("✅ MusicGen BGM 생성 완료:", out_path)
    return out_path

def make_video_segment_from_gif(item):
    gif_path = Path(item["gif_path"])
    out_path = SEGMENT_VIDEO_DIR / f"{item['index']:02d}_{item['requested_action']}.mp4"

    if out_path.exists():
        return out_path

    vf = (
        "fps=12,"
        "scale=512:512:force_original_aspect_ratio=decrease,"
        "pad=512:512:(ow-iw)/2:(oh-ih)/2:color=white,"
        "format=yuv420p"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", str(gif_path),
        "-vf", vf,
        "-an",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        str(out_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1200:])
        raise RuntimeError(f"segment 생성 실패: {gif_path}")

    return out_path

def make_tts_segment(index, scenario):
    out_path = SEGMENT_TTS_DIR / f"{index:02d}_{scenario['name']}.wav"

    if out_path.exists():
        return out_path

    print(f"▶️ TTS 생성: {scenario['name']}")

    generator = get_tts_wav(
        ref_wav_path=str(REF_WAV),
        prompt_text=medium_prompt,
        prompt_language="ko",
        text=scenario["text"],
        text_language="ko",
    )

    sr, audio_data = next(generator)
    sf.write(str(out_path), audio_data, sr)

    return out_path

def mix_one_segment(index, video_path, tts_path, bgm_path):
    out_path = SEGMENT_MIX_DIR / f"{index:02d}_mixed.mp4"

    video_duration = get_duration(video_path)
    tts_duration = get_duration(tts_path)
    target_duration = max(video_duration, tts_duration + 0.4, 3.0)

    filter_complex = (
        f"[0:v]trim=0:{target_duration},setpts=PTS-STARTPTS[v];"
        f"[1:a]volume=1.35,apad,atrim=0:{target_duration}[tts];"
        f"[2:a]volume=-13dB,atrim=0:{target_duration}[bgm];"
        "[tts][bgm]amix=inputs=2:duration=longest:dropout_transition=0:normalize=0[aout]"
    )

    cmd = [
        "ffmpeg", "-y",
        "-stream_loop", "-1",
        "-i", str(video_path),
        "-i", str(tts_path),
        "-stream_loop", "-1",
        "-i", str(bgm_path),
        "-filter_complex", filter_complex,
        "-map", "[v]",
        "-map", "[aout]",
        "-t", str(target_duration),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "veryfast",
        "-crf", "23",
        "-c:a", "aac",
        str(out_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f"segment mix 실패: {index}")

    return out_path

def concat_segments(segment_paths, output_path):
    concat_file = SEGMENT_MIX_DIR / "concat.txt"
    concat_file.write_text(
        "\n".join([f"file '{p.as_posix()}'" for p in segment_paths]),
        encoding="utf-8",
    )

    cmd = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(concat_file),
        "-c", "copy",
        str(output_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print("copy concat 실패. 재인코딩으로 재시도합니다.")
        cmd = [
            "ffmpeg", "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", str(concat_file),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", "veryfast",
            "-crf", "23",
            "-c:a", "aac",
            str(output_path),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("최종 concat 실패")

    return output_path

items = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
valid_items = [item for item in items if item.get("gif_path") and Path(item["gif_path"]).exists()]

assert valid_items, "성공한 gif_path가 없습니다."

ensure_musicgen_bgm(BGM_WAV)

mixed_segments = []
segment_records = []

for i, item in enumerate(valid_items, 1):
    scenario = tts_scenarios[(i - 1) % len(tts_scenarios)]

    print(f"\n▶️ Segment {i}: {item['requested_action']} + {scenario['name']}")

    video_seg = make_video_segment_from_gif(item)
    tts_seg = make_tts_segment(i, scenario)
    mixed_seg = mix_one_segment(i, video_seg, tts_seg, BGM_WAV)

    mixed_segments.append(mixed_seg)
    segment_records.append({
        "index": i,
        "action": item["requested_action"],
        "actual_action": item.get("action"),
        "fallback_used": item.get("fallback_used"),
        "tts_name": scenario["name"],
        "tts_text": scenario["text"],
        "video_segment": str(video_seg),
        "tts_segment": str(tts_seg),
        "mixed_segment": str(mixed_seg),
    })

FINAL_MP4 = concat_segments(mixed_segments, FINAL_MP4)

summary_out = ROOT_OUTPUT / "segment_synced_summary.json"
summary_out.write_text(json.dumps(segment_records, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n✅ 최종 segment-sync 영상:", FINAL_MP4)
display(Video(str(FINAL_MP4), embed=True, width=640))

 여기서는 MusicGen 파일을 표준 WAV로 고쳐 저장한 뒤 FFmpeg에 넣어야 합니다. 특히 이전 코드의 scipy.io.wavfile.write(..., data=audio_values.numpy()) 방식은 MusicGen 텐서가 3차원이라 FFmpeg가 제대로 못 읽거나 믹싱이 이상해질 수 있습니다.



In [ ]:
# A-1. MusicGen BGM 생성/복구: FFmpeg 친화 WAV로 저장
import os, gc, subprocess
from pathlib import Path

import torch
import numpy as np
import soundfile as sf
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from IPython.display import Audio, display

RAW_BGM = Path("/kaggle/working/musicgen_raw.wav")
BGM_WAV = Path("/kaggle/working/musicgen_travel_bgm_fixed.wav")

def run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("명령 실행 실패")
    return result

if not BGM_WAV.exists():
    print("▶️ MusicGen BGM 생성 중...")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

    prompt = "A cheerful acoustic guitar travel vlog instrumental, happy, bright, clean background music"
    inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=750)

    sr = model.config.audio_encoder.sampling_rate

    # 핵심: [batch, channel, samples] -> 1D mono audio
    audio_np = audio_values[0, 0].detach().cpu().float().numpy()
    peak = np.max(np.abs(audio_np))
    if peak > 0:
        audio_np = audio_np / peak * 0.85

    sf.write(str(RAW_BGM), audio_np, sr)

    # 핵심: FFmpeg가 안정적으로 읽는 표준 WAV로 변환
    run([
        "ffmpeg", "-y",
        "-i", str(RAW_BGM),
        "-ac", "2",
        "-ar", "44100",
        "-c:a", "pcm_s16le",
        str(BGM_WAV),
    ])

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✅ BGM 준비 완료:", BGM_WAV)
display(Audio(str(BGM_WAV)))

In [ ]:
# 최종 musicgen_tts_animation
# 애니메이션 + TTS + MusicGen 
# A-2. 애니메이션 + TTS + MusicGen 합성
import subprocess
from pathlib import Path
from IPython.display import Video, display

ANIMATION_MP4 = Path("/kaggle/working/six_doodle_actions_fallback/combined_sequential.mp4")
TTS_WAV = Path("/kaggle/working/final_tts_Vlog_Opening_Excited.wav")
BGM_WAV = Path("/kaggle/working/musicgen_travel_bgm_fixed.wav")
FINAL_MP4 = Path("/kaggle/working/animation_musicgen_tts_fixed.mp4")

assert ANIMATION_MP4.exists(), f"애니메이션 없음: {ANIMATION_MP4}"
assert TTS_WAV.exists(), f"TTS 없음: {TTS_WAV}"
assert BGM_WAV.exists(), f"MusicGen BGM 없음: {BGM_WAV}"

filter_complex = (
    "[1:a]volume=1.35,apad[tts];"
    "[2:a]volume=-13dB,aloop=loop=-1:size=2e+09[bgm_loop];"
    "[tts][bgm_loop]amix=inputs=2:duration=first:dropout_transition=0:normalize=0[aout]"
)

cmd = [
    "ffmpeg", "-y",
    "-i", str(ANIMATION_MP4),
    "-i", str(TTS_WAV),
    "-i", str(BGM_WAV),
    "-filter_complex", filter_complex,
    "-map", "0:v",
    "-map", "[aout]",
    "-c:v", "copy",
    "-c:a", "aac",
    "-shortest",
    str(FINAL_MP4),
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✅ 최종 합성 완료:", FINAL_MP4)
    display(Video(str(FINAL_MP4), embed=True, width=640))
else:
    print("❌ FFmpeg 합성 실패")
    print(result.stderr[-3000:])

In [ ]:
import os
import mlflow
import dagshub
from pathlib import Path

FINAL_MP4 = Path("/kaggle/working/animation_musicgen_tts_segment_synced.mp4")
SUMMARY = Path("/kaggle/working/six_doodle_actions_fallback/segment_synced_summary.json")
ROOT_OUTPUT = Path("/kaggle/working/six_doodle_actions_fallback")

dagshub.init(repo_owner="myelin24m", repo_name="Kride.mlflow", mlflow=True)
mlflow.set_experiment("animated-drawings-segment-audio-sync")

with mlflow.start_run(run_name="segment-synced-animation-musicgen-tts"):
    mlflow.log_param("sync_mode", "per_animation_segment_tts")
    mlflow.log_param("bgm", "musicgen_travel_bgm.wav")
    mlflow.log_metric("segment_count", 6)

    if FINAL_MP4.exists():
        mlflow.log_artifact(str(FINAL_MP4), artifact_path="final_video")

    if SUMMARY.exists():
        mlflow.log_artifact(str(SUMMARY), artifact_path="summary")

    for wav in (ROOT_OUTPUT / "synced_tts").glob("*.wav"):
        mlflow.log_artifact(str(wav), artifact_path="segment_tts")

    print("✅ DagsHub 기록 완료")
    print("DagsHub URI:", mlflow.get_tracking_uri())

C. zrok에 올릴 서버



In [ ]:
%%writefile /kaggle/working/media_app.py
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, FileResponse

app = FastAPI(title="K-Ride Media Preview")

ANIM_BGM = Path("/kaggle/working/animation_with_musicgen.mp4")
FULL = Path("/kaggle/working/animation_with_musicgen_tts.mp4")

@app.get("/", response_class=HTMLResponse)
def index():
    return """
    <html>
      <head><title>K-Ride Preview</title></head>
      <body style="font-family:Arial;padding:24px">
        <h1>K-Ride Media Preview</h1>

        <h2>1. Animation + MusicGen</h2>
        <video src="/video/animation-musicgen" controls width="720"></video>
        <p><a href="/video/animation-musicgen">Download animation + musicgen</a></p>

        <h2>2. Animation + MusicGen + TTS</h2>
        <video src="/video/full" controls width="720"></video>
        <p><a href="/video/full">Download full video</a></p>
      </body>
    </html>
    """

@app.get("/health")
def health():
    return {
        "animation_musicgen_exists": ANIM_BGM.exists(),
        "full_exists": FULL.exists(),
    }

@app.get("/video/animation-musicgen")
def video_animation_musicgen():
    if not ANIM_BGM.exists():
        raise HTTPException(status_code=404, detail="animation_with_musicgen.mp4 not found")
    return FileResponse(str(ANIM_BGM), media_type="video/mp4", filename=ANIM_BGM.name)

@app.get("/video/full")
def video_full():
    if not FULL.exists():
        raise HTTPException(status_code=404, detail="animation_with_musicgen_tts.mp4 not found")
    return FileResponse(str(FULL), media_type="video/mp4", filename=FULL.name)

!pip install -q fastapi uvicorn


In [ ]:
import subprocess, time, requests, os

os.system("pkill -f media_app.py")

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "media_app:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd="/kaggle/working",
    stdout=open("/kaggle/working/media_app.log", "a"),
    stderr=subprocess.STDOUT,
)

time.sleep(5)
print(requests.get("http://127.0.0.1:7860/health").json())

 zrok은 서버를 영구 호스팅하는 서비스라기보다, 지금 실행 중인 Kaggle/로컬 서버를 외부 HTTPS URL로 터널링하는 도구


 배포 노트북 분리

Deploy_A_Static_Demo: char1~char6 렌더링, 합본 GIF/MP4 생성, DagsHub 기록
Deploy_B_API_zrok: FastAPI 서버 실행, zrok으로 외부 공개
Experiment_CogVideoX, Experiment_GPT-SoVITS는 별도 실험 노트북으로 유지
첫 배포 목표

TorchServe 없는 안정 버전부터 성공시키기
/examples/characters/char1~char6 + 6개 motion
개별 GIF 6개 + 합본 GIF 생성
DagsHub MLflow artifact 업로드
zrok으로 결과 페이지 또는 API 공개
zrok 공개 방식 선택

결과물만 보여주기: --backend-mode web 정적 페이지 공유
API까지 열기: FastAPI localhost:8001을 zrok proxy로 공유




D. zrok 설치 + 공개 URL 생성



In [ ]:
import os, json, urllib.request, subprocess, shutil, stat, time
from pathlib import Path

ZROK_BIN = Path("/kaggle/working/zrok")
DOWNLOAD_DIR = Path("/kaggle/working/zrok_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not ZROK_BIN.exists():
    meta = json.load(urllib.request.urlopen("https://api.github.com/repos/openziti/zrok/releases/latest"))
    assets = meta["assets"]

    candidates = [
        a for a in assets
        if "linux_amd64" in a["name"] and a["name"].endswith(".tar.gz")
    ]

    if not candidates:
        raise RuntimeError("linux_amd64 zrok release asset을 찾지 못했습니다.")

    asset = candidates[0]
    url = asset["browser_download_url"]
    tgz = DOWNLOAD_DIR / asset["name"]

    print("▶️ zrok 다운로드:", url)
    subprocess.run(["curl", "-L", url, "-o", str(tgz)], check=True)

    extract_dir = DOWNLOAD_DIR / "extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(["tar", "-xzf", str(tgz), "-C", str(extract_dir)], check=True)

    found = []
    for path in extract_dir.rglob("*"):
        if path.is_file() and path.name in {"zrok", "zrok2"}:
            found.append(path)

    if not found:
        print("압축 해제 파일 목록:")
        for path in extract_dir.rglob("*"):
            print(path)
        raise FileNotFoundError("zrok 또는 zrok2 실행파일을 찾지 못했습니다.")

    shutil.copy(found[0], ZROK_BIN)
    ZROK_BIN.chmod(ZROK_BIN.stat().st_mode | stat.S_IEXEC)

subprocess.run([str(ZROK_BIN), "version"], check=False)

처음 한 번만 zrok 토큰을 활성화하세요.



In [ ]:
import getpass, subprocess

ZROK_TOKEN = getpass.getpass("zrok enable token: ")
subprocess.run([str(ZROK_BIN), "enable", ZROK_TOKEN], check=True)

In [ ]:
import subprocess, time, re
from pathlib import Path

zrok_log = open("/kaggle/working/zrok_share.log", "w")

share = subprocess.Popen(
    [str(ZROK_BIN), "share", "public", "http://127.0.0.1:7860", "--headless"],
    stdout=zrok_log,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(8)
zrok_log.close()

log_text = Path("/kaggle/working/zrok_share.log").read_text(errors="ignore")
print(log_text[-3000:])

urls = re.findall(r"https://[^\s]+", log_text)
print("zrok public urls:", urls)

In [ ]:
import os, json, urllib.request, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
ZROK_BIN = WORK / "zrok2"

if not ZROK_BIN.exists():
    meta = json.load(urllib.request.urlopen("https://api.github.com/repos/openziti/zrok/releases/latest"))
    tag = meta["tag_name"]
    version = tag.lstrip("v")
    url = f"https://github.com/openziti/zrok/releases/download/{tag}/zrok_{version}_linux_amd64.tar.gz"

    print("▶️ zrok 다운로드:", url)
    subprocess.run(f"curl -L '{url}' -o {WORK}/zrok.tgz", shell=True, check=True)

    extract_dir = WORK / "zrok_extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(f"tar -xzf {WORK}/zrok.tgz -C {extract_dir}", shell=True, check=True)

    print("압축 해제 파일 목록:")
    for x in extract_dir.rglob("*"):
        print(" -", x)

    candidates = [
        x for x in extract_dir.rglob("*")
        if x.is_file() and x.name in ("zrok", "zrok2")
    ]

    if not candidates:
        raise FileNotFoundError("zrok/zrok2 실행파일을 압축 해제 결과에서 찾지 못했습니다.")

    shutil.copy(candidates[0], ZROK_BIN)
    os.chmod(ZROK_BIN, 0o755)
    print("✅ zrok 설치 완료:", ZROK_BIN)
else:
    print("✅ zrok already exists:", ZROK_BIN)

subprocess.run([str(ZROK_BIN), "version"], check=False)

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient
print("zrok 설치 오류 수정 셀")
ZROK = "/kaggle/working/zrok"
ZROK_TOKEN = UserSecretsClient().get_secret("ZROK_TOKEN")

result = subprocess.run(
    [ZROK, "enable", ZROK_TOKEN],
    capture_output=True,
    text=True,
)

print(result.stdout[-1000:])
print(result.stderr[-1000:])

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient
pring(" zrok enable 셀")
ZROK_BIN = "/kaggle/working/zrok2"
ZROK_TOKEN = UserSecretsClient().get_secret("ZROK_TOKEN")

result = subprocess.run(
    [ZROK_BIN, "enable", ZROK_TOKEN],
    capture_output=True,
    text=True,
)

print(result.stdout[-1000:])
print(result.stderr[-1000:])

In [ ]:
from pathlib import Path
import shutil, json, os

print(" zrok에 올릴 정적 페이지 만들기")

PUBLIC_DIR = Path("/kaggle/working/public")
PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

combined = Path("/kaggle/working/six_doodle_actions_fallback/combined_2x3.gif")
summary = Path("/kaggle/working/six_doodle_actions_fallback/render_summary.json")

if combined.exists():
    shutil.copy(combined, PUBLIC_DIR / "combined_2x3.gif")
if summary.exists():
    shutil.copy(summary, PUBLIC_DIR / "render_summary.json")

html_text = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>K-Ride Animated Doodles</title>
  <style>
    body { font-family: Arial, sans-serif; padding: 28px; background: #f7f7f7; color: #222; }
    main { max-width: 980px; margin: 0 auto; }
    img { max-width: 100%; border: 1px solid #ddd; background: white; }
    .note { margin-top: 12px; color: #555; }
  </style>
</head>
<body>
  <main>
    <h1>K-Ride Animated Doodles</h1>
    <p>Six doodle characters rendered with motion fallback handling.</p>
    <img src="combined_2x3.gif" alt="Combined animation grid">
    <p class="note">Fallback is applied when a character is missing required skeleton joints for the requested motion.</p>
  </main>
</body>
</html>
"""
(PUBLIC_DIR / "index.html").write_text(html_text, encoding="utf-8")

print("✅ public page ready:", PUBLIC_DIR)

In [ ]:
import subprocess, time, re
from pathlib import Path
print('정적 페이지 서버 + zrok share')
PUBLIC_DIR = "/kaggle/working/public"
ZROK_BIN = "/kaggle/working/zrok2"

# 정적 파일 서버
static_log = open("/kaggle/working/static_server.log", "w")
static_proc = subprocess.Popen(
    ["python", "-m", "http.server", "8088", "--directory", PUBLIC_DIR],
    stdout=static_log,
    stderr=subprocess.STDOUT,
)

time.sleep(3)

# zrok public share
zrok_log_path = Path("/kaggle/working/zrok_share.log")
zrok_proc = subprocess.Popen(
    [ZROK_BIN, "share", "public", "http://127.0.0.1:8088", "--headless"],
    stdout=open(zrok_log_path, "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(10)
log = zrok_log_path.read_text(errors="ignore")
print(log[-4000:])

urls = re.findall(r"https://[a-zA-Z0-9.-]+", log)
print("🌐 zrok URL:", urls[-1] if urls else "URL을 로그에서 확인하세요")

In [ ]:
# zrok 배포 + DagsHub 기록 통합 셀
import os, re, json, time, shutil, socket, signal, subprocess, urllib.request, tarfile, getpass
from pathlib import Path

import mlflow
import dagshub
import requests

# ------------------------------------------------------------
# 0. 최종 배포할 MP4 선택
# ------------------------------------------------------------
CANDIDATES = [
    Path("/kaggle/working/animation_musicgen_tts_segment_synced.mp4"),
    Path("/kaggle/working/animation_musicgen_tts_fixed.mp4"),
    Path("/kaggle/working/final_travel_vlog_with_MLflow.mp4"),
    Path("/kaggle/working/final_travel_vlog.mp4"),
    Path("/kaggle/working/six_doodle_actions_fallback/combined_sequential.mp4"),
]

FINAL_VIDEO = next((p for p in CANDIDATES if p.exists()), None)
assert FINAL_VIDEO is not None, "배포할 MP4를 찾지 못했습니다. 먼저 최종 합성 MP4를 생성해주세요."

print("✅ 배포 대상:", FINAL_VIDEO)

# ------------------------------------------------------------
# 1. 필요 패키지 설치
# ------------------------------------------------------------
subprocess.run(
    ["pip", "install", "-q", "fastapi", "uvicorn", "mlflow", "dagshub", "requests"],
    check=False,
)

# ------------------------------------------------------------
# 2. zrok 다운로드/설치
# ------------------------------------------------------------
WORK = Path("/kaggle/working")
ZROK_BIN = WORK / "zrok"

if not ZROK_BIN.exists():
    print("▶️ zrok 다운로드 중...")

    meta = json.load(urllib.request.urlopen("https://api.github.com/repos/openziti/zrok/releases/latest"))
    tag = meta["tag_name"]
    version = tag.lstrip("v")
    url = f"https://github.com/openziti/zrok/releases/download/{tag}/zrok_{version}_linux_amd64.tar.gz"

    tgz = WORK / "zrok.tgz"
    extract_dir = WORK / "zrok_extract"

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(["curl", "-L", url, "-o", str(tgz)], check=True)

    with tarfile.open(tgz, "r:gz") as tar:
        tar.extractall(extract_dir)

    found = []
    for p in extract_dir.rglob("*"):
        if p.is_file() and p.name in ["zrok", "zrok2"]:
            found.append(p)

    assert found, "zrok 실행파일을 압축 해제 결과에서 찾지 못했습니다."

    shutil.copy(found[0], ZROK_BIN)
    os.chmod(ZROK_BIN, 0o755)

subprocess.run([str(ZROK_BIN), "version"], check=False)

# ------------------------------------------------------------
# 3. zrok enable
# ------------------------------------------------------------
zrok_env = Path.home() / ".zrok" / "environment.json"

if not zrok_env.exists():
    token = os.environ.get("ZROK_TOKEN")
    if not token:
        token = getpass.getpass("🔑 zrok enable token 입력: ")

    print("▶️ zrok enable 실행 중...")
    result = subprocess.run(
        [str(ZROK_BIN), "enable", token],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("zrok enable 실패")
else:
    print("✅ zrok environment가 이미 활성화되어 있습니다.")

# ------------------------------------------------------------
# 4. FastAPI 서버 파일 생성
# ------------------------------------------------------------
DEPLOY_DIR = WORK / "kride_zrok_deploy"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

APP_FILE = DEPLOY_DIR / "app.py"
APP_FILE.write_text(f'''
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, FileResponse

VIDEO_PATH = Path(r"{str(FINAL_VIDEO)}")

app = FastAPI(title="K-Ride Media Preview")

@app.get("/health")
def health():
    return {{
        "ok": VIDEO_PATH.exists(),
        "video_path": str(VIDEO_PATH),
        "video_size_mb": round(VIDEO_PATH.stat().st_size / 1024 / 1024, 2) if VIDEO_PATH.exists() else 0,
    }}

@app.get("/", response_class=HTMLResponse)
def index():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")

    return f"""
    <!doctype html>
    <html>
    <head>
      <meta charset="utf-8">
      <title>K-Ride Preview</title>
      <style>
        body {{
          margin: 0;
          font-family: Arial, sans-serif;
          background: #111;
          color: #fff;
          display: flex;
          min-height: 100vh;
          align-items: center;
          justify-content: center;
        }}
        main {{
          width: min(960px, 94vw);
        }}
        video {{
          width: 100%;
          background: #000;
        }}
        a {{
          color: #8fd3ff;
        }}
      </style>
    </head>
    <body>
      <main>
        <h2>K-Ride AI Travel Vlog Preview</h2>
        <video src="/video.mp4" controls autoplay loop></video>
        <p><a href="/download">Download MP4</a></p>
      </main>
    </body>
    </html>
    """

@app.get("/video.mp4")
def video():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")
    return FileResponse(str(VIDEO_PATH), media_type="video/mp4")

@app.get("/download")
def download():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")
    return FileResponse(str(VIDEO_PATH), media_type="video/mp4", filename=VIDEO_PATH.name)
''', encoding="utf-8")

# ------------------------------------------------------------
# 5. 이전 프로세스 종료: 이 셀이 만든 PID만 종료
# ------------------------------------------------------------
def stop_pid_file(pid_file):
    pid_file = Path(pid_file)
    if pid_file.exists():
        try:
            pid = int(pid_file.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
            print("✅ 이전 프로세스 종료:", pid)
        except Exception as e:
            print("⚠️ 이전 프로세스 종료 건너뜀:", e)
        pid_file.unlink(missing_ok=True)

SERVER_PID = WORK / "kride_fastapi.pid"
ZROK_PID = WORK / "kride_zrok.pid"

stop_pid_file(SERVER_PID)
stop_pid_file(ZROK_PID)

# ------------------------------------------------------------
# 6. 사용 가능한 포트 찾고 FastAPI 실행
# ------------------------------------------------------------
sock = socket.socket()
sock.bind(("127.0.0.1", 0))
PORT = sock.getsockname()[1]
sock.close()

SERVER_LOG = WORK / "kride_fastapi.log"
server_log_f = open(SERVER_LOG, "w")

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=str(DEPLOY_DIR),
    stdout=server_log_f,
    stderr=subprocess.STDOUT,
)

SERVER_PID.write_text(str(server_proc.pid))
print("▶️ FastAPI 서버 시작:", f"http://127.0.0.1:{PORT}")

time.sleep(5)

health = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).json()
print("✅ health:", health)

# ------------------------------------------------------------
# 7. zrok public share 실행
# zrok 공식 문서 기준: zrok share public --headless 8080 형태
# ------------------------------------------------------------
ZROK_LOG = WORK / "kride_zrok.log"
zrok_log_f = open(ZROK_LOG, "w")

zrok_proc = subprocess.Popen(
    [str(ZROK_BIN), "share", "public", "--headless", str(PORT)],
    stdout=zrok_log_f,
    stderr=subprocess.STDOUT,
)

ZROK_PID.write_text(str(zrok_proc.pid))
print("▶️ zrok share 시작. 공개 URL 감지 중...")

public_url = None
for _ in range(40):
    time.sleep(1)
    zrok_log_f.flush()

    text = ZROK_LOG.read_text(errors="ignore") if ZROK_LOG.exists() else ""
    urls = re.findall(r"https?://[a-zA-Z0-9\\-\\.]+\\.share\\.zrok\\.io[^\\s]*", text)
    if urls:
        public_url = urls[-1]
        break

if not public_url:
    print("⚠️ zrok URL을 자동 감지하지 못했습니다. 로그 마지막 부분:")
    print(ZROK_LOG.read_text(errors="ignore")[-3000:])
    raise RuntimeError("zrok public URL 감지 실패")

print("🎉 zrok 배포 URL:", public_url)

# ------------------------------------------------------------
# 8. DagsHub / MLflow 기록
# ------------------------------------------------------------
dagshub.init(repo_owner="myelin24m", repo_name="Kride.mlflow", mlflow=True)
mlflow.set_experiment("kride-zrok-deployment")

if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="zrok-public-video-deployment") as run:
    mlflow.log_param("public_url", public_url)
    mlflow.log_param("local_url", f"http://127.0.0.1:{PORT}")
    mlflow.log_param("final_video_path", str(FINAL_VIDEO))
    mlflow.log_param("zrok_command", f"zrok share public --headless {PORT}")
    mlflow.log_metric("video_size_mb", FINAL_VIDEO.stat().st_size / 1024 / 1024)
    mlflow.log_metric("deployment_port", PORT)
    mlflow.log_metric("zrok_url_detected", 1)

    mlflow.log_artifact(str(FINAL_VIDEO), artifact_path="deployed_video")
    mlflow.log_artifact(str(APP_FILE), artifact_path="deployment_app")
    mlflow.log_artifact(str(SERVER_LOG), artifact_path="deployment_logs")
    mlflow.log_artifact(str(ZROK_LOG), artifact_path="deployment_logs")

    print("✅ DagsHub 기록 완료")
    print("Run ID:", run.info.run_id)
    print("MLflow URI:", mlflow.get_tracking_uri())
    print("Public URL:", public_url)

print("\n배포는 이 노트북 세션과 zrok 프로세스가 살아있는 동안 유지됩니다.")

In [ ]:
# zrok URL 재감지 + DagsHub 기록 복구 셀
import re, time
from pathlib import Path

import requests
import mlflow
import dagshub

ZROK_LOG = Path("/kaggle/working/kride_zrok.log")
FINAL_VIDEO = Path("/kaggle/working/animation_musicgen_tts_segment_synced.mp4")

assert ZROK_LOG.exists(), f"zrok 로그 없음: {ZROK_LOG}"
assert FINAL_VIDEO.exists(), f"최종 영상 없음: {FINAL_VIDEO}"

log_text = ZROK_LOG.read_text(errors="ignore")
print(log_text[-3000:])

# zrok v1/v2 모두 대응:
# - https://xxx.share.zrok.io
# - xxx.share.zrok.io
# - xxx.shares.zrok.io
matches = re.findall(
    r"(https?://[a-zA-Z0-9.-]+\.shares?\.zrok\.io|[a-zA-Z0-9.-]+\.shares?\.zrok\.io)",
    log_text,
)

if not matches:
    raise RuntimeError("zrok URL을 로그에서 찾지 못했습니다.")

public_url = matches[-1]
if not public_url.startswith("http"):
    public_url = "https://" + public_url

print("🎉 zrok public URL:", public_url)

# 접속 테스트
for path in ["/health", "/"]:
    try:
        r = requests.get(public_url + path, timeout=20)
        print(path, r.status_code)
    except Exception as e:
        print(path, "접속 테스트 실패:", e)

# DagsHub / MLflow 기록
dagshub.init(repo_owner="myelin24m", repo_name="Kride.mlflow", mlflow=True)
mlflow.set_experiment("kride-zrok-deployment")

if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="zrok-public-video-deployment-recovered") as run:
    mlflow.log_param("public_url", public_url)
    mlflow.log_param("final_video_path", str(FINAL_VIDEO))
    mlflow.log_param("zrok_log_path", str(ZROK_LOG))
    mlflow.log_metric("video_size_mb", FINAL_VIDEO.stat().st_size / 1024 / 1024)
    mlflow.log_metric("zrok_url_detected", 1)

    mlflow.log_artifact(str(FINAL_VIDEO), artifact_path="deployed_video")
    mlflow.log_artifact(str(ZROK_LOG), artifact_path="deployment_logs")

    print("✅ DagsHub 기록 완료")
    print("Run ID:", run.info.run_id)
    print("MLflow URI:", mlflow.get_tracking_uri())
    print("Public URL:", public_url)

옵션B

In [ ]:
import subprocess, time, requests, os

# media_server.py가 /kaggle/working에 있다고 가정
server_log = open("/kaggle/working/server_log.txt", "w")

server_proc = subprocess.Popen(
    ["python", "-u", "/kaggle/working/media_server.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

time.sleep(10)

res = requests.get("http://127.0.0.1:8001/docs", timeout=10)
print("FastAPI status:", res.status_code)

In [ ]:
import subprocess, time, re
from pathlib import Path

ZROK = "/kaggle/working/zrok"
log_path = "/kaggle/working/zrok_api.log"

zrok_proc = subprocess.Popen(
    [ZROK, "share", "public", "8001", "--headless"],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(8)
log = Path(log_path).read_text(errors="ignore")
print(log[-3000:])

urls = re.findall(r"https://[a-zA-Z0-9.-]+", log)
print("🌐 API URL:", urls[-1] if urls else "로그에서 URL을 확인하세요")

In [ ]:
from pathlib import Path
import subprocess, time, requests, os, signal

APP_FILE = Path("/kaggle/working/kride_zrok_deploy/app.py")
FINAL_VIDEO = Path("/kaggle/working/animation_musicgen_tts_segment_synced.mp4")
SERVER_PID = Path("/kaggle/working/kride_fastapi.pid")
SERVER_LOG = Path("/kaggle/working/kride_fastapi.log")
PORT = 42853  # 지금 health가 뜬 포트

APP_FILE.write_text(f'''
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, FileResponse

VIDEO_PATH = Path(r"{str(FINAL_VIDEO)}")

app = FastAPI(title="K-Ride Media Preview")

@app.get("/health")
def health():
    return {{
        "ok": VIDEO_PATH.exists(),
        "video_path": str(VIDEO_PATH),
        "video_size_mb": round(VIDEO_PATH.stat().st_size / 1024 / 1024, 2) if VIDEO_PATH.exists() else 0,
    }}

@app.get("/", response_class=HTMLResponse)
def index():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")

    return """
    <!doctype html>
    <html>
    <head>
      <meta charset="utf-8">
      <title>K-Ride Preview</title>
      <style>
        body {{
          margin: 0;
          font-family: Arial, sans-serif;
          background: #111;
          color: #fff;
          display: flex;
          min-height: 100vh;
          align-items: center;
          justify-content: center;
        }}
        main {{
          width: min(960px, 94vw);
        }}
        video {{
          width: 100%;
          background: #000;
        }}
        a {{
          color: #8fd3ff;
        }}
      </style>
    </head>
    <body>
      <main>
        <h2>K-Ride AI Travel Vlog Preview</h2>
        <video src="/video.mp4" controls autoplay loop></video>
        <p><a href="/download">Download MP4</a></p>
      </main>
    </body>
    </html>
    """

@app.get("/video.mp4")
def video():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")
    return FileResponse(str(VIDEO_PATH), media_type="video/mp4")

@app.get("/download")
def download():
    if not VIDEO_PATH.exists():
        raise HTTPException(status_code=404, detail="video not found")
    return FileResponse(str(VIDEO_PATH), media_type="video/mp4", filename=VIDEO_PATH.name)
''', encoding="utf-8")

# 기존 FastAPI 서버 종료
if SERVER_PID.exists():
    try:
        pid = int(SERVER_PID.read_text().strip())
        os.kill(pid, signal.SIGTERM)
        time.sleep(2)
        print("✅ 기존 서버 종료:", pid)
    except Exception as e:
        print("⚠️ 기존 서버 종료 실패/무시:", e)

# 서버 재시작
server_log_f = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd="/kaggle/working/kride_zrok_deploy",
    stdout=server_log_f,
    stderr=subprocess.STDOUT,
)

SERVER_PID.write_text(str(server_proc.pid))
time.sleep(5)

print("local health:", requests.get(f"http://127.0.0.1:{PORT}/health").status_code)
print(requests.get(f"http://127.0.0.1:{PORT}/health").text)

root = requests.get(f"http://127.0.0.1:{PORT}/")
print("local root:", root.status_code)
print(root.text[:300])

In [ ]:
import requests

PUBLIC_URL = "https://koqjqv72xmyz.shares.zrok.io"

print("public health:", requests.get(PUBLIC_URL + "/health", timeout=20).status_code)
print("public root:", requests.get(PUBLIC_URL + "/", timeout=20).status_code)
print("public video:", requests.get(PUBLIC_URL + "/video.mp4", timeout=20).status_code)

In [ ]:
ZROK = "/kaggle/working/zrok"
UNIQUE_NAME = "kride24demo"

# 한 번만 실행
subprocess.run([ZROK, "reserve", "public", "8001", "--unique-name", UNIQUE_NAME], check=False)

# 이후 실행마다
subprocess.Popen(
    [ZROK, "share", "reserved", UNIQUE_NAME, "--headless"],
    stdout=open("/kaggle/working/zrok_reserved.log", "w"),
    stderr=subprocess.STDOUT,
)

In [ ]:
import requests, subprocess
from pathlib import Path

PUBLIC_URL = "https://koqjqv72xmyz.shares.zrok.io"

print("1) local health")
try:
    print(requests.get("http://127.0.0.1:42853/health", timeout=10).status_code)
    print(requests.get("http://127.0.0.1:42853/health", timeout=10).text[:1000])
except Exception as e:
    print("local health error:", e)

print("\n2) public health")
try:
    print(requests.get(PUBLIC_URL + "/health", timeout=20).status_code)
    print(requests.get(PUBLIC_URL + "/health", timeout=20).text[:1000])
except Exception as e:
    print("public health error:", e)

print("\n3) local root")
try:
    r = requests.get("http://127.0.0.1:42853/", timeout=10)
    print(r.status_code)
    print(r.text[:1000])
except Exception as e:
    print("local root error:", e)

print("\n4) FastAPI log")
for log_path in [
    "/kaggle/working/kride_fastapi.log",
    "/kaggle/working/media_app.log",
]:
    p = Path(log_path)
    if p.exists():
        print(f"\n--- {log_path} ---")
        print(p.read_text(errors="ignore")[-4000:])

In [ ]:
# 1. Kaggle 작업물 백업 zip 생성
import os, json, shutil, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP_DIR = WORK / f"kride_backup_{STAMP}"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

files = [
    WORK / "animation_musicgen_tts_segment_synced.mp4",
    WORK / "animation_musicgen_tts_fixed.mp4",
    WORK / "animation_with_musicgen.mp4",
    WORK / "animation_with_musicgen_tts.mp4",
    WORK / "final_tts_Vlog_Opening_Excited.wav",
    WORK / "musicgen_travel_bgm_fixed.wav",
    WORK / "musicgen_travel_bgm.wav",
    WORK / "media_app.py",
    WORK / "kride_fastapi.log",
    WORK / "kride_zrok.log",
    WORK / "zrok_share.log",
]

dirs = [
    WORK / "six_doodle_actions_fallback",
    WORK / "kride_zrok_deploy",
    WORK / "public",
]

copied = []

for f in files:
    if f.exists():
        dst = BACKUP_DIR / f.name
        shutil.copy2(f, dst)
        copied.append(str(f))

for d in dirs:
    if d.exists():
        dst = BACKUP_DIR / d.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(d, dst)
        copied.append(str(d))

manifest = {
    "created_at": STAMP,
    "copied_items": copied,
    "note": "K-Ride Kaggle backup. Tokens and large environments are intentionally excluded.",
}

(BACKUP_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

zip_path = shutil.make_archive(str(BACKUP_DIR), "zip", BACKUP_DIR)

print("✅ 백업 완료:", zip_path)
print("다운로드 대상:", zip_path)

In [ ]:
# 2. DagsHub / MLflow에 백업 zip과 최종 산출물 기록
import mlflow
import dagshub
from pathlib import Path

dagshub.init(repo_owner="myelin24m", repo_name="Kride.mlflow", mlflow=True)
mlflow.set_experiment("kride-backup-and-handoff")

if mlflow.active_run():
    mlflow.end_run()

zip_path = Path(zip_path)

with mlflow.start_run(run_name="kaggle-session-backup") as run:
    mlflow.log_param("backup_zip", str(zip_path))
    mlflow.log_metric("backup_size_mb", zip_path.stat().st_size / 1024 / 1024)
    mlflow.log_artifact(str(zip_path), artifact_path="kaggle_backup")

    print("✅ DagsHub 백업 기록 완료")
    print("Run ID:", run.info.run_id)
    print("MLflow URI:", mlflow.get_tracking_uri())

In [ ]:
# 실행한 notebook cell 코드 백업
import os, sys, json, time, shutil, subprocess
from pathlib import Path
from IPython import get_ipython

WORK = Path("/kaggle/working")
STAMP = time.strftime("%Y%m%d_%H%M%S")
CODE_BACKUP = WORK / f"kride_code_backup_{STAMP}"
CODE_BACKUP.mkdir(parents=True, exist_ok=True)

ip = get_ipython()
inputs = ip.user_ns.get("In", [])

# 1. 실행 셀을 .py로 백업
py_path = CODE_BACKUP / "notebook_cells_backup.py"
with open(py_path, "w", encoding="utf-8") as f:
    f.write("# K-Ride Kaggle executed notebook cells backup\n")
    f.write(f"# created_at: {STAMP}\n\n")
    for i, cell in enumerate(inputs):
        if not cell or not cell.strip():
            continue
        f.write(f"\n# %% [cell {i}]\n")
        f.write(cell)
        f.write("\n")

# 2. 실행 셀을 .ipynb로도 백업
ipynb = {
    "cells": [],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {"name": "python"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

for i, cell in enumerate(inputs):
    if not cell or not cell.strip():
        continue
    ipynb["cells"].append({
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": cell.splitlines(True),
    })

ipynb_path = CODE_BACKUP / "notebook_cells_backup.ipynb"
ipynb_path.write_text(json.dumps(ipynb, ensure_ascii=False, indent=2), encoding="utf-8")

# 3. requirements / pip freeze 백업
with open(CODE_BACKUP / "requirements_freeze.txt", "w", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, stderr=subprocess.STDOUT)

with open(CODE_BACKUP / "pip_list.txt", "w", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "list"], stdout=f, stderr=subprocess.STDOUT)

# 4. 직접 만든 주요 코드 파일 백업
for path in [
    WORK / "media_app.py",
    WORK / "worker.py",
    WORK / "run_tts.py",
    WORK / "kride_zrok_deploy" / "app.py",
]:
    if path.exists():
        dst = CODE_BACKUP / path.name
        if path.name == "app.py":
            dst = CODE_BACKUP / "zrok_app.py"
        shutil.copy2(path, dst)

# 5. 실행 결과 요약 파일 백업
for path in [
    WORK / "six_doodle_actions_fallback" / "render_summary.json",
    WORK / "six_doodle_actions_fallback" / "segment_synced_summary.json",
]:
    if path.exists():
        shutil.copy2(path, CODE_BACKUP / path.name)

manifest = {
    "created_at": STAMP,
    "python": sys.version,
    "backup_dir": str(CODE_BACKUP),
    "notes": [
        "Executed cell history was saved from IPython In[] history.",
        "Secrets/tokens are not intentionally included.",
        "Large model folders are excluded; restore them with setup cells.",
    ],
}

(CODE_BACKUP / "code_backup_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

zip_path = shutil.make_archive(str(CODE_BACKUP), "zip", CODE_BACKUP)

print("✅ 코드 백업 완료:", zip_path)